
# Experimento COBYLA — versão 15

## Objetivo desta versão

Antes de alterar o treinamento do Transformer, vamos responder duas perguntas.

### Pergunta 1 — o melhor tempo de aquecimento depende do orçamento?

Será comparado:

```text
0 + 250   COBYLA completo
10 + 240  Top-4 curto + refinamento completo
20 + 230  Top-4 intermediário + refinamento completo
30 + 220  Top-4 maior + refinamento completo
40 + 210  configuração já testada
```

Todos os métodos começam do mesmo ponto inicial e possuem orçamento total máximo de 250 avaliações.

### Pergunta 2 — os parâmetros sensíveis mudam quando $n$ e $k$ mudam?

O teste principal usa dez ativos e varia:

$$
k\in\{2,3,4,5\}.
$$

Também existe um modo opcional para variar o número de ativos:

$$
(n,k)
\in
\{(6,2),(6,3),(8,2),(8,3),(8,4)\}.
$$

Para cada configuração será produzido um pequeno banco de vetores otimizados e uma nova análise de sensibilidade.

A quantidade de parâmetros do ansatz não é fixa. Para esta construção de Dicke:

$$
m(n,k)
=
\sum_{l=0}^{n-1}\min(k,l)
=
\frac{k(2n-k-1)}{2}.
$$

Por exemplo:

- $m(10,2)=17$;
- $m(10,3)=24$;
- $m(10,4)=30$;
- $m(10,5)=35$.

O objetivo não é procurar sempre os índices `2, 3, 14, 17`. O objetivo é descobrir se as regiões sensíveis ocupam posições estruturais semelhantes no circuito quando $n$ e $k$ mudam.



## Dependências

Execute esta instalação apenas quando os pacotes não estiverem disponíveis:

```python
# %pip install --upgrade \
#     numpy pandas matplotlib cvxpy pyscipopt \
#     docplex qiskit qiskit-aer \
#     qiskit-optimization qiskit-algorithms
```

Depois da instalação, reinicie o kernel.


In [ ]:

# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
from time import perf_counter
from typing import Any, Iterable
import json
import os
import math
import warnings

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from docplex.mp.model import Model

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.primitives import BackendEstimatorV2
from qiskit_aer import AerSimulator

from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import (
        NumPyMinimumEigensolver,
    )
    from qiskit_algorithms.optimizers import COBYLA
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import (
        NumPyMinimumEigensolver,
    )
    from qiskit.algorithms.optimizers import COBYLA


# ------------------------------------------------------------
# Problema fixo
# ------------------------------------------------------------

stock_path = Path("../data/assets/stocks_prices.csv")

n_assets = 10
budget = 4

q_value = 0.5
risk_free = 0.0475

known_exact_bitstring = "1001011000"
known_exact_energy = -0.8181062577496281

# ------------------------------------------------------------
# Configuração do experimento
# ------------------------------------------------------------

fast_mode = True

n_restarts = 3 if fast_mode else 20
maxiter_main = 250 if fast_mode else 1000
maxiter_refinement = 80 if fast_mode else 300

shots = 4096
random_seed = 42
near_optimal_threshold = 1e-3

run_dimension_curve = True
run_two_stage = True

# ------------------------------------------------------------
# DIRETÓRIO DE RESULTADOS COM CAMINHO CURTO
# ------------------------------------------------------------
#
# No Windows, caminhos tradicionais podem ultrapassar o limite
# aceito por algumas bibliotecas e produzir:
#
# WinError 206:
# "O nome do arquivo ou a extensão é muito grande."
#
# Para evitar isso, os resultados são gravados fora da pasta
# profundamente aninhada do projeto:
#
# Windows:
#     C:\Users\<usuario>\vqe_r
#
# Linux/macOS:
#     ./vqe_r
#
# É possível definir outro diretório antes de executar o notebook:
#
# Windows PowerShell:
#     $env:VQE_RESULTS_DIR = "D:\vqe_r"
#
# Linux/macOS:
#     export VQE_RESULTS_DIR="/tmp/vqe_r"

custom_results_root = os.environ.get(
    "VQE_RESULTS_DIR"
)

if custom_results_root:
    results_root = Path(
        custom_results_root
    ).expanduser()
elif os.name == "nt":
    results_root = (
        Path.home()
        / "vqe_r"
    )
else:
    results_root = Path(
        "vqe_r"
    )

# Nome curto para não somar muitos caracteres aos arquivos internos.
output_dir = (
    results_root
    / "base"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "diretório curto de resultados:",
    output_dir.resolve(),
)

print(
    "comprimento do caminho-base:",
    len(
        str(
            output_dir.resolve()
        )
    ),
)

rng = np.random.default_rng(
    random_seed
)

print("arquivo de preços:", stock_path)
print("problema:", f"{n_assets} ativos, escolher {budget}")
print("q:", q_value)
print("taxa livre de risco:", risk_free)
print("bitstring conhecido:", known_exact_bitstring)
print("modo rápido:", fast_mode)



# Parte I — reconstrução idêntica do problema original

O problema financeiro é reconstruído uma única vez. Durante todo o experimento:

- os preços não mudam;
- os retornos não mudam;
- a covariância não muda;
- o QUBO não muda;
- o Hamiltoniano de Ising não muda;
- o ansatz não muda.

Somente o conjunto de parâmetros liberados para o COBYLA é alterado.


In [ ]:

# ============================================================
# 2. PREÇOS -> RETORNOS DIÁRIOS -> RETORNO DOS ATIVOS
# ============================================================

if not stock_path.exists():
    raise FileNotFoundError(
        "O arquivo não foi encontrado em:\n"
        f"{stock_path.resolve()}"
    )

stocks_prices = pd.read_csv(
    "../data/assets/stocks_prices.csv",
    index_col=0,
)

# Garante que somente dados numéricos permaneçam.
stocks_prices = (
    stocks_prices
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna(
        axis=1,
        how="all",
    )
    .ffill()
    .bfill()
    .dropna()
)

if stocks_prices.shape[1] != n_assets:
    raise ValueError(
        f"Esperados {n_assets} ativos, mas o CSV possui "
        f"{stocks_prices.shape[1]} colunas numéricas."
    )

# CÓDIGO ORIGINAL:
# daily_returns = stocks_prices.pct_change().dropna()
daily_returns = (
    stocks_prices
    .pct_change(
        fill_method=None
    )
    .dropna()
)

# PONTO FUNDAMENTAL:
# O notebook original usa a SOMA dos retornos diários.
# Não trocar por mean() neste experimento, pois isso mudaria
# o problema, o Ising, a energia e possivelmente o bitstring.
assets_returns = daily_returns.sum()

# Matriz de covariância exatamente como no original.
covariance = daily_returns.cov()

# Simetrização numérica sem alterar a formulação.
covariance_values = covariance.to_numpy(
    dtype=float
)
covariance_values = 0.5 * (
    covariance_values
    + covariance_values.T
)

assets_return_values = assets_returns.to_numpy(
    dtype=float
)

tickers = list(
    stocks_prices.columns
)

print("tickers:", tickers)
print("preços:", stocks_prices.shape)
print("retornos diários:", daily_returns.shape)
print("retornos acumulados:", assets_returns.shape)
print("covariância:", covariance.shape)

display(
    assets_returns.to_frame(
        "assets_return"
    )
)

display(covariance)


In [ ]:

# ============================================================
# 3. PROBLEMA CLÁSSICO COM CVXPY
# ============================================================

# Parâmetro original de ponderação risco-retorno.
q_cvx = cp.Parameter(
    nonneg=True,
    name="q",
)
q_cvx.value = q_value

# Uma variável booleana para cada ativo:
# 0 = não selecionado
# 1 = selecionado
var_x = cp.Variable(
    shape=(covariance.shape[0],),
    boolean=True,
    name="portfolio_selection",
)

# OBJETIVO ORIGINAL:
#
# q * risco
# - (1-q) * retorno
# + taxa livre de risco
#
# O termo risk_free é constante. Ele desloca a energia,
# mas não altera qual carteira minimiza a função.
objective_cvx = cp.Minimize(
    q_cvx
    * cp.quad_form(
        var_x,
        cp.psd_wrap(
            covariance_values
        ),
    )
    - (1 - q_cvx)
    * var_x
    @ assets_return_values
    + risk_free
)

# Restrição: selecionar exatamente quatro ativos.
constraints_cvx = [
    np.ones(
        shape=(n_assets,)
    )
    @ var_x
    == budget
]

problem_cvx = cp.Problem(
    objective=objective_cvx,
    constraints=constraints_cvx,
)

cvxpy_status = None
cvxpy_solution = None
cvxpy_value = None

installed_solvers = set(
    cp.installed_solvers()
)

if "SCIP" in installed_solvers:
    cvxpy_value = problem_cvx.solve(
        solver="SCIP"
    )
    cvxpy_status = problem_cvx.status

    if var_x.value is not None:
        cvxpy_solution = np.rint(
            np.asarray(
                var_x.value
            ).reshape(-1)
        ).astype(int)
else:
    warnings.warn(
        "O SCIP não está disponível no CVXPY. "
        "A enumeração exata das 210 carteiras será usada "
        "como validação clássica."
    )

print("status CVXPY:", cvxpy_status)
print("valor CVXPY:", cvxpy_value)
print("solução CVXPY:", cvxpy_solution)


In [ ]:

# ============================================================
# 4. ENUMERAÇÃO EXATA DAS C(10,4)=210 CARTEIRAS
# ============================================================

def original_classical_objective(
    x_binary,
):
    """
    Mesma função objetivo utilizada em CVXPY e DOcplex.
    """
    x_binary = np.asarray(
        x_binary,
        dtype=float,
    ).reshape(-1)

    risk_term = (
        q_value
        * x_binary
        @ covariance_values
        @ x_binary
    )

    return_term = (
        (1 - q_value)
        * x_binary
        @ assets_return_values
    )

    return float(
        risk_term
        - return_term
        + risk_free
    )


enumeration_rows = []

for selected_indices in combinations(
    range(n_assets),
    budget,
):
    x_candidate = np.zeros(
        n_assets,
        dtype=int,
    )
    x_candidate[
        list(selected_indices)
    ] = 1

    bitstring_asset_order = "".join(
        str(int(value))
        for value in x_candidate
    )

    # O Qiskit exibe q_(n-1)...q_0.
    bitstring_qiskit_order = (
        bitstring_asset_order[::-1]
    )

    enumeration_rows.append(
        {
            "selected_indices": selected_indices,
            "x_asset_order": x_candidate,
            "bitstring_asset_order": bitstring_asset_order,
            "bitstring_qiskit_order": bitstring_qiskit_order,
            "objective": original_classical_objective(
                x_candidate
            ),
        }
    )

enumeration_df = (
    pd.DataFrame(
        enumeration_rows
    )
    .sort_values(
        "objective"
    )
    .reset_index(
        drop=True
    )
)

enumeration_best = enumeration_df.iloc[0]

classical_exact_value = float(
    enumeration_best[
        "objective"
    ]
)

classical_exact_x = np.asarray(
    enumeration_best[
        "x_asset_order"
    ],
    dtype=int,
)

classical_asset_bitstring = str(
    enumeration_best[
        "bitstring_asset_order"
    ]
)

classical_qiskit_bitstring = str(
    enumeration_best[
        "bitstring_qiskit_order"
    ]
)

selected_tickers = [
    ticker
    for ticker, selected
    in zip(
        tickers,
        classical_exact_x,
    )
    if selected == 1
]

print("carteiras avaliadas:", len(enumeration_df))
print("valor clássico exato:", classical_exact_value)
print("vetor em ordem dos ativos:", classical_exact_x)
print("bitstring em ordem dos ativos:", classical_asset_bitstring)
print("bitstring em ordem Qiskit:", classical_qiskit_bitstring)
print("ativos selecionados:", selected_tickers)

display(
    enumeration_df[
        [
            "selected_indices",
            "bitstring_asset_order",
            "bitstring_qiskit_order",
            "objective",
        ]
    ].head(10)
)


In [ ]:

# ============================================================
# 5. DOCPLEX -> QUADRATIC PROGRAM -> QUBO -> ISING
# ============================================================

model = Model(
    name="portfolio_optimization"
)

variables = np.array(
    [
        model.binary_var(
            name=f"x_{index}"
        )
        for index in range(n_assets)
    ],
    dtype=object,
)

# No código original a expressão foi escrita de forma compacta:
#
# variables @ covariance.values @ variables
#
# A soma explícita abaixo é matematicamente idêntica, mas evita
# problemas entre versões do NumPy e objetos simbólicos do DOcplex.
risk_expression = model.sum(
    float(
        covariance_values[i, j]
    )
    * variables[i]
    * variables[j]
    for i in range(n_assets)
    for j in range(n_assets)
)

return_expression = model.sum(
    float(
        assets_return_values[i]
    )
    * variables[i]
    for i in range(n_assets)
)

model.minimize(
    q_value
    * risk_expression
    - (1 - q_value)
    * return_expression
    + risk_free
)

model.add_constraint(
    model.sum(
        variables.tolist()
    )
    == budget,
    ctname="budget_exactly_four",
)

quad_model = from_docplex_mp(
    model=model
)

qubo_converter = (
    QuadraticProgramToQubo()
)

qubo = qubo_converter.convert(
    quad_model
)

ising, offset = qubo.to_ising()

print("QuadraticProgram:")
print(quad_model.prettyprint())

print()
print("QUBO:")
print(qubo.prettyprint())

print()
print("qubits do Ising:", ising.num_qubits)
print("offset:", offset)
print("termos de Pauli:", len(ising.paulis))


In [ ]:

# ============================================================
# 6. SOLUÇÃO EXATA DO HAMILTONIANO
# ============================================================

exact_solver = (
    NumPyMinimumEigensolver()
)

exact_result = (
    exact_solver
    .compute_minimum_eigenvalue(
        operator=ising
    )
)

exact_energy = float(
    np.real(
        exact_result.eigenvalue
        + offset
    )
)

exact_state_dict = (
    exact_result
    .eigenstate
    .to_dict()
)

exact_bitstring = str(
    max(
        exact_state_dict,
        key=lambda key: abs(
            exact_state_dict[key]
        ) ** 2,
    )
).replace(
    " ",
    "",
)

print("energia exata Ising + offset:", exact_energy)
print("bitstring exato do Ising:", exact_bitstring)
print("energia esperada do código original:", known_exact_energy)
print("bitstring esperado:", known_exact_bitstring)

energy_matches = np.isclose(
    exact_energy,
    known_exact_energy,
    atol=1e-10,
    rtol=0.0,
)

bitstring_matches = (
    exact_bitstring
    == known_exact_bitstring
)

print("energia coincide?", energy_matches)
print("bitstring coincide?", bitstring_matches)

if not energy_matches:
    raise RuntimeError(
        "A energia reconstruída não coincide com a energia "
        "do problema original. Verifique preços, retorno=sum(), "
        "q=0.5, risk_free=0.0475 e período dos dados."
    )

if not bitstring_matches:
    raise RuntimeError(
        "O bitstring reconstruído não coincide com 1001011000. "
        "Não execute o teste dos theta antes de corrigir "
        "a reconstrução do problema."
    )

EXACT_ENERGY = exact_energy
EXACT_BITSTRING = exact_bitstring



# Parte II — ansatz de Dicke parametrizado embutido no notebook

As três funções usadas no código original agora estão definidas diretamente neste arquivo:

```python
CY_parameterized(...)
CCY_parameterized(...)
DickeStateAnsatz(n, k)
```

Isso elimina completamente a dependência de:

```text
01_generate_dataset.ipynb
```

## Onde `DickeStateAnsatz` é chamada

No teste tradicional do código original:

```python
ansatz = DickeStateAnsatz(
    n=ising.num_qubits,
    k=4,
)
ansatz = ansatz.decompose()
```

Na geração das rodadas:

```python
ansatz_base = DickeStateAnsatz(
    n=ising.num_qubits,
    k=4,
).decompose()
```

Neste experimento será criado **somente `ansatz_base`**, uma única vez. Todas as estratégias — COBYLA completo, Top-1, Top-2, Top-4 e controles — usam exatamente esse mesmo circuito.

> Isso é importante porque os nomes simbólicos dos parâmetros incluem um sufixo aleatório. Recriar o ansatz no meio do experimento pode produzir novos nomes e alterar a correspondência entre a posição do vetor e a porta física. Por isso, a semente é fixada antes da construção e o circuito-base é congelado para toda a comparação.


In [ ]:

# ============================================================
# 7. PORTAS PARAMETRIZADAS E DickeStateAnsatz
#    IMPLEMENTAÇÃO EMBUTIDA — SEM DEPENDER DE OUTRO NOTEBOOK
# ============================================================

def CY_parameterized(i):
    """
    Cria a porta parametrizada chamada CY no código original.

    Parâmetro
    ---------
    i:
        Identificador usado somente para gerar um nome simbólico
        único para o ângulo da porta.

    Funcionamento
    -------------
    A porta interna é uma rotação controlada em Y:
        controle = qubit 1
        alvo     = qubit 0

    O valor numérico do ângulo só será inserido mais tarde por
    `assign_parameters`.
    """
    param = ParameterVector(
        name=f"x{i}",
        length=1,
    )

    qc = QuantumCircuit(2)

    qc.cry(
        param[0],
        1,
        0,
    )

    gate = qc.to_gate(
        label="CY"
    )

    return gate


def CCY_parameterized(i):
    """
    Cria a porta parametrizada chamada CCY no código original.

    Ela usa:
    1. uma rotação RY;
    2. uma porta Toffoli CCX;
    3. a rotação inversa;
    4. uma segunda CCX.

    O parâmetro continua simbólico até a chamada de
    `assign_parameters`.
    """
    param = ParameterVector(
        name=f"y{i}",
        length=1,
    )

    qc = QuantumCircuit(3)

    qc.ry(
        param[0],
        0,
    )

    qc.ccx(
        2,
        1,
        0,
    )

    qc.ry(
        -param[0],
        0,
    )

    qc.ccx(
        2,
        1,
        0,
    )

    gate = qc.to_gate(
        label="CCY"
    )

    return gate


def DickeStateAnsatz(n, k):
    """
    Constrói o ansatz parametrizado de Dicke usado no código original.

    Parâmetros
    ----------
    n:
        Número total de qubits.

    k:
        Peso de Hamming fixo. Neste problema:
            n = 10
            k = 4

    Ideia física/combinatória
    -------------------------
    Os últimos k qubits começam em |1>, formando um estado inicial
    com exatamente k excitações. As portas CY e CCY redistribuem
    amplitudes dentro do subespaço de peso de Hamming k.

    Consequência
    ------------
    A busca fica concentrada nas C(10,4)=210 soluções viáveis,
    em vez de explorar indiscriminadamente os 2^10 estados.
    """
    qr = QuantumRegister(
        n,
        "q",
    )

    qc = QuantumCircuit(
        qr
    )

    # --------------------------------------------------------
    # Inicialização:
    # aplica X nos últimos k qubits.
    #
    # Para n=10 e k=4, começa em:
    # |0000001111> na convenção lógica do registrador.
    # --------------------------------------------------------
    for i in range(k):
        qc.x(
            n - i - 1
        )

    aux = 1

    # Percorre os qubits do final para o início.
    for l in range(n)[::-1]:

        # Para cada l, percorre até k posições anteriores.
        for i in range(
            l - 1,
            l - 1 - k,
            -1,
        ):
            if i >= 0:

                # O código original cria um nome praticamente único
                # concatenando:
                #   l, i, contador auxiliar e inteiro aleatório.
                #
                # Esse nome identifica o parâmetro simbólico da porta.
                a = (
                    str(l)
                    + str(i)
                    + str(aux)
                    + str(
                        np.random.randint(
                            0,
                            1e8,
                        )
                    )
                )

                if i == l - 1:
                    # Caso vizinho:
                    # CX -> CY parametrizada -> CX
                    qc.cx(
                        qr[i],
                        qr[l],
                    )

                    qc.append(
                        CY_parameterized(a),
                        [
                            qr[i],
                            qr[l],
                        ],
                    )

                    qc.cx(
                        qr[i],
                        qr[l],
                    )

                else:
                    # Caso não vizinho:
                    # CX -> CCY parametrizada -> CX
                    #
                    # Mantém a mesma forma do código original,
                    # inclusive o uso de índices inteiros em qc.cx.
                    qc.cx(
                        i,
                        l,
                    )

                    qc.append(
                        CCY_parameterized(a),
                        [
                            qr[i],
                            qr[i + 1],
                            qr[l],
                        ],
                    )

                    qc.cx(
                        i,
                        l,
                    )

            # O contador avança em toda passagem do laço interno,
            # exatamente como na construção original.
            aux += 1

    return qc


# ------------------------------------------------------------
# REPRODUTIBILIDADE DA ORDEM DOS PARÂMETROS
# ------------------------------------------------------------
#
# Os identificadores das portas contêm np.random.randint(...).
# A semente é fixada imediatamente antes de criar o ansatz.
#
# IMPORTANTE:
# não execute novamente somente esta parte durante uma comparação
# já iniciada. Recrie o kernel e execute o notebook em ordem.
parameter_name_seed = random_seed

np.random.seed(
    parameter_name_seed
)

print(
    "CY_parameterized definida:",
    callable(CY_parameterized),
)

print(
    "CCY_parameterized definida:",
    callable(CCY_parameterized),
)

print(
    "DickeStateAnsatz definida:",
    callable(DickeStateAnsatz),
)


In [ ]:

# ============================================================
# 8. CHAMADA DO DickeStateAnsatz E CONGELAMENTO DO ANSATZ BASE
# ============================================================

# No teste tradicional do código original aparecia:
#
# ansatz = DickeStateAnsatz(
#     n=ising.num_qubits,
#     k=4,
# )
# ansatz = ansatz.decompose()
#
# Na geração das 10.000 rodadas aparecia:
#
# ansatz_base = DickeStateAnsatz(
#     n=ising.num_qubits,
#     k=4,
# ).decompose()
#
# Neste experimento usamos apenas `ansatz_base`.
# Ele é criado uma única vez e compartilhado por todos os métodos.

ansatz_base = DickeStateAnsatz(
    n=ising.num_qubits,
    k=budget,
).decompose()

n_parameters = int(
    ansatz_base.num_parameters
)

if n_parameters != 30:
    raise RuntimeError(
        "O ansatz deveria possuir 30 parâmetros para "
        f"n=10 e k=4, mas possui {n_parameters}."
    )

# O Qiskit associa uma sequência de 30 números à ordem abaixo.
# Essa tabela é essencial para interpretar:
# theta_0, theta_1, ..., theta_29.
parameter_order = list(
    ansatz_base.parameters
)

parameter_index_df = pd.DataFrame(
    {
        "theta_index": np.arange(
            n_parameters
        ),
        "parameter_name": [
            str(parameter)
            for parameter
            in parameter_order
        ],
    }
)

parameter_index_df.to_csv(
    output_dir
    / "theta_map.csv",
    index=False,
)

print(
    "qubits do ansatz:",
    ansatz_base.num_qubits,
)

print(
    "parâmetros simbólicos:",
    n_parameters,
)

print(
    "ansatz criado uma única vez e congelado para todo o experimento."
)

display(
    parameter_index_df
)



# Auditoria do mapeamento físico dos $\theta$

O objetivo desta auditoria é responder:

> O parâmetro chamado $\theta_{17}$ no banco original atua sobre a mesma porta e os mesmos qubits que o índice 17 deste experimento?

Comparar apenas o nome simbólico não é suficiente, porque a função original acrescenta um número aleatório ao nome de cada parâmetro. A comparação correta usa uma **assinatura física**, formada por:

- posição da instrução no circuito;
- tipo da porta;
- qubits envolvidos;
- posição do parâmetro dentro da porta;
- sinal/coficiente com que o parâmetro aparece.

A auditoria possui três níveis:

1. **autoteste:** reconstrói o mesmo ansatz com duas sementes e verifica se os índices mudam;
2. **exportação do circuito atual:** salva o mapa físico deste experimento;
3. **comparação com o circuito original:** lê o mapa exportado no kernel que gerou o banco e traduz os índices originais para os índices atuais.


In [ ]:

# ============================================================
# AUDITORIA A. CONSTRUIR O MAPA FÍSICO DE CADA THETA
# ============================================================

import json


def _parameter_linear_coefficient(expression, parameter):
    """
    Obtém o coeficiente linear com que um parâmetro aparece
    em uma expressão de porta.

    Exemplos:
        theta      -> +1
        -theta     -> -1
        2 * theta  -> +2
    """
    try:
        value_zero = expression.bind({parameter: 0.0})
        value_one = expression.bind({parameter: 1.0})
        return float(value_one - value_zero)
    except Exception:
        try:
            value_zero = expression.assign(parameter, 0.0)
            value_one = expression.assign(parameter, 1.0)
            return float(value_one - value_zero)
        except Exception:
            return np.nan


def build_theta_physical_map(circuit):
    """
    Cria um mapa theta_index -> assinatura física.

    A assinatura não usa o nome aleatório do parâmetro.
    Ela descreve onde o parâmetro atua no circuito.
    """
    ordered_parameters = list(circuit.parameters)

    qubit_index = {
        qubit: index
        for index, qubit in enumerate(circuit.qubits)
    }

    uses = {
        parameter: []
        for parameter in ordered_parameters
    }

    for instruction_index, instruction in enumerate(circuit.data):
        operation = instruction.operation

        operation_qubits = [
            int(qubit_index[qubit])
            for qubit in instruction.qubits
        ]

        for parameter_slot, expression in enumerate(operation.params):
            expression_parameters = getattr(
                expression,
                "parameters",
                set(),
            )

            for parameter in expression_parameters:
                if parameter not in uses:
                    continue

                coefficient = _parameter_linear_coefficient(
                    expression,
                    parameter,
                )

                uses[parameter].append(
                    {
                        "instruction_index": int(instruction_index),
                        "operation": str(operation.name),
                        "qubits": operation_qubits,
                        "parameter_slot": int(parameter_slot),
                        "coefficient": (
                            None
                            if not np.isfinite(coefficient)
                            else round(float(coefficient), 12)
                        ),
                    }
                )

    rows = []

    for theta_index, parameter in enumerate(ordered_parameters):
        occurrences = uses[parameter]

        physical_signature = json.dumps(
            occurrences,
            sort_keys=True,
            separators=(",", ":"),
        )

        rows.append(
            {
                "theta_index": int(theta_index),
                "parameter_name": str(parameter),
                "n_occurrences": int(len(occurrences)),
                "physical_signature": physical_signature,
                "occurrences_readable": json.dumps(
                    occurrences,
                    indent=2,
                    ensure_ascii=False,
                ),
            }
        )

    mapping_df = pd.DataFrame(rows)

    if mapping_df["physical_signature"].duplicated().any():
        duplicated = mapping_df[
            mapping_df["physical_signature"].duplicated(keep=False)
        ]

        raise RuntimeError(
            "Foram encontradas assinaturas físicas repetidas. "
            "A correspondência não é unívoca.\n"
            + duplicated[
                [
                    "theta_index",
                    "parameter_name",
                    "physical_signature",
                ]
            ].to_string(index=False)
        )

    return mapping_df


current_theta_map = build_theta_physical_map(ansatz_base)

current_theta_map.to_csv(
    output_dir / "exp_map.csv",
    index=False,
)

display(
    current_theta_map[
        [
            "theta_index",
            "parameter_name",
            "n_occurrences",
            "physical_signature",
        ]
    ]
)


In [ ]:

# ============================================================
# AUDITORIA B. AUTOTESTE COM DUAS SEMENTES
# ============================================================


def build_ansatz_with_seed(seed):
    """
    Reconstrói temporariamente o ansatz preservando
    o estado aleatório global do notebook.
    """
    numpy_random_state = np.random.get_state()

    try:
        np.random.seed(int(seed))
        temporary_ansatz = DickeStateAnsatz(
            n=ising.num_qubits,
            k=budget,
        ).decompose()
    finally:
        np.random.set_state(numpy_random_state)

    return temporary_ansatz


audit_ansatz_seed_a = build_ansatz_with_seed(random_seed)
audit_ansatz_seed_b = build_ansatz_with_seed(random_seed + 1001)

audit_map_seed_a = build_theta_physical_map(audit_ansatz_seed_a)
audit_map_seed_b = build_theta_physical_map(audit_ansatz_seed_b)

# A junção é feita pela assinatura física, e não pelo nome.
seed_comparison_df = audit_map_seed_a.merge(
    audit_map_seed_b,
    on="physical_signature",
    suffixes=("_seed_a", "_seed_b"),
    validate="one_to_one",
)

seed_comparison_df["same_theta_index"] = (
    seed_comparison_df["theta_index_seed_a"]
    == seed_comparison_df["theta_index_seed_b"]
)

same_index_fraction = float(
    seed_comparison_df["same_theta_index"].mean()
)

print(
    "parâmetros físicos comparados:",
    len(seed_comparison_df),
)

print(
    "fração que manteve o mesmo índice:",
    f"{same_index_fraction:.2%}",
)

display(
    seed_comparison_df[
        [
            "theta_index_seed_a",
            "parameter_name_seed_a",
            "theta_index_seed_b",
            "parameter_name_seed_b",
            "same_theta_index",
            "physical_signature",
        ]
    ].sort_values("theta_index_seed_a")
)

if same_index_fraction < 1.0:
    print(
        "\nRESULTADO DA AUDITORIA:"
        "\nOs nomes aleatórios alteram a associação "
        "índice -> porta física entre construções."
        "\nOs índices do banco original precisam ser traduzidos."
    )
else:
    print(
        "\nRESULTADO DA AUDITORIA:"
        "\nNesta versão do circuito, a ordem física permaneceu estável "
        "entre as duas sementes testadas."
    )



## Exportar o mapa do circuito que gerou as 10.000 execuções

A etapa decisiva deve ser executada **no notebook/kernel original**, onde está o `ansatz_base` utilizado na geração do banco.

Copie para o notebook original:

1. a célula `AUDITORIA A`, que define `build_theta_physical_map`;
2. depois execute:

```python
original_theta_map = build_theta_physical_map(
    ansatz_base
)

original_theta_map.to_csv(
    "../results/orig_map.csv",
    index=False,
)

display(
    original_theta_map
)
```

O arquivo precisa ser salvo **antes de reiniciar o kernel original**, caso esse circuito ainda esteja em memória.

Se o kernel original já foi perdido e a semente/estado do NumPy usado naquela construção não foi registrado, não é possível provar retrospectivamente qual porta física correspondia a cada posição do vetor `best_parameters`. Nesse caso, o ranking deve ser reconstruído usando um ansatz determinístico.


In [ ]:

# ============================================================
# AUDITORIA C. COMPARAR ORIGINAL X EXPERIMENTO
# ============================================================

original_map_candidates = [
    Path("../results/orig_map.csv"),
    Path("results/orig_map.csv"),
    output_dir / "orig_map.csv",
]

original_map_path = next(
    (
        candidate.resolve()
        for candidate in original_map_candidates
        if candidate.exists()
    ),
    None,
)

theta_translation_df = None

if original_map_path is None:
    print(
        "Mapa original ainda não localizado."
        "\nExporte-o no notebook/kernel que gerou o banco."
    )
else:
    original_theta_map = pd.read_csv(original_map_path)

    required_columns = {
        "theta_index",
        "physical_signature",
    }

    missing_columns = required_columns - set(original_theta_map.columns)

    if missing_columns:
        raise ValueError(
            "O mapa original não possui as colunas: "
            f"{sorted(missing_columns)}"
        )

    theta_translation_df = (
        original_theta_map[
            [
                "theta_index",
                "parameter_name",
                "physical_signature",
            ]
        ]
        .rename(
            columns={
                "theta_index": "theta_index_original",
                "parameter_name": "parameter_name_original",
            }
        )
        .merge(
            current_theta_map[
                [
                    "theta_index",
                    "parameter_name",
                    "physical_signature",
                ]
            ].rename(
                columns={
                    "theta_index": "theta_index_experiment",
                    "parameter_name": "parameter_name_experiment",
                }
            ),
            on="physical_signature",
            how="inner",
            validate="one_to_one",
        )
        .sort_values("theta_index_original")
        .reset_index(drop=True)
    )

    theta_translation_df["same_index"] = (
        theta_translation_df["theta_index_original"]
        == theta_translation_df["theta_index_experiment"]
    )

    if len(theta_translation_df) != n_parameters:
        raise RuntimeError(
            "Nem todas as 30 assinaturas físicas foram encontradas. "
            f"Foram associadas {len(theta_translation_df)}."
        )

    theta_translation_df.to_csv(
        output_dir / "theta_xlat.csv",
        index=False,
    )

    print("mapa original:", original_map_path)
    print(
        "índices que permaneceram iguais:",
        int(theta_translation_df["same_index"].sum()),
        "de",
        n_parameters,
    )

    display(theta_translation_df)

    original_top_indices = [2, 3, 14, 17]

    translated_top_indices_df = (
        theta_translation_df[
            theta_translation_df["theta_index_original"].isin(
                original_top_indices
            )
        ][
            [
                "theta_index_original",
                "theta_index_experiment",
                "parameter_name_original",
                "parameter_name_experiment",
                "physical_signature",
            ]
        ]
        .sort_values("theta_index_original")
    )

    print("\nTradução dos índices prioritários:")
    display(translated_top_indices_df)



## O que `assign_parameters` faz

O circuito `ansatz_base` contém portas com parâmetros simbólicos. Por exemplo, internamente podem existir objetos equivalentes a:

```text
x... [0]
y... [0]
```

Depois que o COBYLA encontra valores numéricos, esta linha:

```python
ansatz_copy = (
    ansatz_base
    .copy()
    .assign_parameters(theta_completo)
)
```

substitui cada parâmetro simbólico pelo valor correspondente de `theta_completo`.

Ela **não otimiza**, **não calcula energia** e **não altera o Hamiltoniano**. Ela apenas cria a versão numérica do circuito para a simulação e a medição.

### Diferença crucial no experimento reduzido

No COBYLA completo:

```python
result_optimizer.x.shape == (30,)
```

No Top-4:

```python
result_optimizer.x.shape == (4,)
```

Portanto, isto estaria errado:

```python
# ERRADO no experimento Top-4:
ansatz_base.assign_parameters(
    result_optimizer.x
)
```

Antes, é obrigatório reconstruir:


$$
\theta_{\mathrm{completo}}
=
(\theta_0,\ldots,\theta_{29}),
$$


inserindo os quatro valores otimizados e preservando os outros 26 valores fixos.


In [ ]:

# ============================================================
# 9. SIMULADOR, ESTIMADOR E FUNÇÃO DE ENERGIA ORIGINAL
# ============================================================

simulator = AerSimulator(
    method="statevector",
    device="CPU",
)

estimator = BackendEstimatorV2(
    backend=simulator
)

# O callback é reiniciado antes de cada otimização.
callback_list = []


def scalarize_energy(
    value: Any,
) -> float:
    """
    Converte saídas NumPy escalares ou arrays de um elemento em float.
    """
    return float(
        np.real(
            np.asarray(
                value
            ).reshape(-1)[0]
        )
    )


def exp_val(
    x: np.ndarray,
) -> float:
    """
    Função objetivo usada pelo COBYLA.

    Esta implementação preserva a lógica do código original:
    1. copia o ansatz;
    2. decompõe o circuito;
    3. envia circuito, Ising e vetor theta ao Estimator;
    4. adiciona o offset;
    5. salva theta e energia no callback.
    """
    global callback_list

    x = np.asarray(
        x,
        dtype=float,
    ).reshape(-1)

    if x.size != n_parameters:
        raise ValueError(
            "exp_val sempre exige o vetor COMPLETO com "
            f"{n_parameters} parâmetros; recebeu {x.size}."
        )

    ansatz_copy = (
        ansatz_base
        .copy()
        .decompose()
    )

    estimator_result = estimator.run(
        pubs=[
            (
                ansatz_copy,
                ising,
                x,
            )
        ]
    ).result()

    cost_function_value = scalarize_energy(
        estimator_result[0].data.evs
    )

    original_energy = float(
        cost_function_value
        + offset
    )

    callback_list.append(
        (
            x.copy(),
            original_energy,
        )
    )

    return original_energy


print(
    "função exp_val pronta para",
    n_parameters,
    "parâmetros",
)


In [ ]:

# ============================================================
# 10. AUDITORIA EXPLÍCITA DO assign_parameters
# ============================================================

def assign_full_parameters(
    theta_full: np.ndarray,
) -> QuantumCircuit:
    """
    Converte o ansatz simbólico em circuito numérico.

    IMPORTANTE:
    - theta_full precisa possuir 30 valores;
    - inplace=False preserva ansatz_base para todas as rodadas;
    - a atribuição é posicional e segue parameter_order.
    """
    theta_full = np.asarray(
        theta_full,
        dtype=float,
    ).reshape(-1)

    if theta_full.size != n_parameters:
        raise ValueError(
            "assign_parameters exige o vetor completo: "
            f"esperados {n_parameters}, recebidos {theta_full.size}."
        )

    # RELAÇÃO EXATA COM A LINHA DA FOTO:
    #
    # ansatz_copy = (
    #     ansatz_base
    #     .copy()
    #     .assign_parameters(
    #         result_optimizer.x
    #     )
    # )
    #
    # No experimento reduzido usamos theta_full, pois
    # result_optimizer.x contém somente os parâmetros ativos.
    ansatz_numeric = (
        ansatz_base
        .copy()
        .assign_parameters(
            theta_full,
            inplace=False,
        )
    )

    if ansatz_numeric.num_parameters != 0:
        raise RuntimeError(
            "Ainda restaram parâmetros simbólicos após assign_parameters."
        )

    return ansatz_numeric


# Teste simples com um vetor de 30 valores.
theta_assignment_test = np.zeros(
    n_parameters,
    dtype=float,
)

assigned_test_circuit = (
    assign_full_parameters(
        theta_assignment_test
    )
)

print(
    "parâmetros restantes após atribuição:",
    assigned_test_circuit.num_parameters,
)

print(
    "ansatz_base continua simbólico:",
    ansatz_base.num_parameters,
)

print(
    "cópia atribuída é totalmente numérica:",
    assigned_test_circuit.num_parameters == 0,
)



# Parte III — COBYLA em subespaço ativo

Para um conjunto ativo $S$, o COBYLA recebe somente:


$$
z=(\theta_j)_{j\in S}.
$$


A função objetivo reconstrói o vetor completo:


$$
\theta_j=
\begin{cases}
z_j, & j\in S,\\
\theta_j^{\mathrm{fixo}}, & j\notin S.
\end{cases}
$$


Então `exp_val` continua recebendo exatamente 30 parâmetros, como no código original.


In [ ]:

# ============================================================
# 11. RANKING E CONJUNTOS EXPERIMENTAIS
# ============================================================

# Ranking empírico do maior para o menor contraste
# na taxa do bitstring exato.
ranked_indices = [
    17, 2, 14, 3, 24, 4, 18, 1, 16, 20,
    8, 6, 0, 19, 15, 28, 5, 29, 7, 9,
    23, 25, 10, 21, 13, 11, 12, 22, 26, 27,
]

if sorted(
    ranked_indices
) != list(
    range(n_parameters)
):
    raise RuntimeError(
        "ranked_indices precisa conter 0...29 sem repetição."
    )

def translate_original_theta_indices(original_indices):
    """
    Traduz índices do circuito original para o circuito atual
    usando a assinatura física.

    Sem mapa original disponível, mantém os índices informados,
    mas registra que a correspondência ainda não foi comprovada.
    """
    original_indices = [int(index) for index in original_indices]

    if (
        "theta_translation_df" not in globals()
        or theta_translation_df is None
    ):
        warnings.warn(
            "O mapa físico original não foi carregado. "
            "Os índices serão usados sem tradução."
        )
        return sorted(original_indices)

    translation = dict(
        zip(
            theta_translation_df["theta_index_original"].astype(int),
            theta_translation_df["theta_index_experiment"].astype(int),
        )
    )

    missing = [
        index
        for index in original_indices
        if index not in translation
    ]

    if missing:
        raise RuntimeError(
            "Não foi possível traduzir os índices: "
            f"{missing}"
        )

    return sorted(translation[index] for index in original_indices)


# Estes são os índices encontrados na auditoria do BANCO ORIGINAL.
top1 = translate_original_theta_indices([17])
top2 = translate_original_theta_indices([14, 17])
top4 = translate_original_theta_indices([2, 3, 14, 17])

# Os controles também pertencem ao ranking original.
bottom4 = translate_original_theta_indices([12, 22, 26, 27])

random_pool = [
    index
    for index
    in range(n_parameters)
    if index not in top4
]

random4 = sorted(
    rng.choice(
        random_pool,
        size=4,
        replace=False,
    ).tolist()
)

experiments = {
    "full_30": list(
        range(n_parameters)
    ),
    "top1_theta17": top1,
    "top2_theta14_theta17": top2,
    "top4_theta02_03_14_17": top4,
    "bottom4_control": bottom4,
    "random4_control": random4,
}

display(
    pd.DataFrame(
        [
            {
                "method": method,
                "n_active": len(indices),
                "active_indices": indices,
            }
            for method, indices
            in experiments.items()
        ]
    )
)


In [ ]:

# ============================================================
# 12. PONTOS INICIAIS PAREADOS
# ============================================================

# O código original gera os theta do Dicke no intervalo:
#
# 0.5 * pi * random()
#
# Para uma comparação justa, cada restart possui UM vetor
# inicial completo. Todos os métodos usam exatamente esse
# mesmo vetor no mesmo restart.
initial_points = (
    0.5
    * np.pi
    * rng.random(
        size=(
            n_restarts,
            n_parameters,
        )
    )
)

initial_points_dict = {
    "Dicke_state": [
        row.copy()
        for row in initial_points
    ]
}

print(
    "matriz de pontos iniciais:",
    initial_points.shape,
)

print(
    "intervalo:",
    float(initial_points.min()),
    "até",
    float(initial_points.max()),
)


In [ ]:

# ============================================================
# 13. RECONSTRUÇÃO DO VETOR COMPLETO
# ============================================================

def reconstruct_full_theta(
    theta_active: np.ndarray,
    active_indices: Iterable[int],
    fixed_reference: np.ndarray,
) -> np.ndarray:
    """
    Insere os valores ativos no vetor fixo de 30 dimensões.

    Esta função é a ponte entre:
    - o COBYLA reduzido, que enxerga 1, 2, 4... variáveis;
    - o circuito original, que sempre exige 30 parâmetros.
    """
    active_indices = np.asarray(
        list(active_indices),
        dtype=int,
    )

    theta_active = np.asarray(
        theta_active,
        dtype=float,
    ).reshape(-1)

    fixed_reference = np.asarray(
        fixed_reference,
        dtype=float,
    ).reshape(-1)

    if fixed_reference.size != n_parameters:
        raise ValueError(
            "fixed_reference precisa possuir "
            f"{n_parameters} valores."
        )

    if theta_active.size != active_indices.size:
        raise ValueError(
            "Quantidade de valores ativos diferente "
            "da quantidade de índices ativos."
        )

    theta_full = (
        fixed_reference.copy()
    )

    theta_full[
        active_indices
    ] = theta_active

    return theta_full


In [ ]:

# ============================================================
# 14. MEDIÇÃO FINAL COM assign_parameters
# ============================================================

def sample_final_solution(
    theta_full: np.ndarray,
    seed_simulator: int,
) -> dict[str, Any]:
    """
    Liga os 30 theta ao circuito e executa 4096 shots.

    Esta é a etapa correspondente ao bloco mostrado nas fotos:
        ansatz_base.copy().assign_parameters(...)
        measure_all()
        simulator.run(...).get_counts()
    """
    theta_full = np.asarray(
        theta_full,
        dtype=float,
    ).reshape(-1)

    # AQUI ocorre o assign_parameters.
    # Ele recebe o vetor COMPLETO reconstruído.
    ansatz_copy = (
        assign_full_parameters(
            theta_full
        )
    )

    # A medição é adicionada somente à cópia numérica.
    # ansatz_base permanece simbólico e reutilizável.
    ansatz_copy.measure_all()

    simulator_start = perf_counter()

    counts = (
        simulator
        .run(
            ansatz_copy,
            shots=shots,
            seed_simulator=seed_simulator,
        )
        .result()
        .get_counts()
    )

    simulator_time = (
        perf_counter()
        - simulator_start
    )

    counts_clean = {
        str(key).replace(
            " ",
            "",
        ): int(value)
        for key, value
        in counts.items()
    }

    total_counts = int(
        sum(
            counts_clean.values()
        )
    )

    most_frequent_bitstring = max(
        counts_clean,
        key=counts_clean.get,
    )

    exact_count = counts_clean.get(
        EXACT_BITSTRING,
        0,
    )

    exact_probability = (
        exact_count
        / total_counts
        if total_counts > 0
        else np.nan
    )

    return {
        "counts": counts_clean,
        "total_counts": total_counts,
        "most_frequent_bitstring": most_frequent_bitstring,
        "probability_best_answer": exact_probability,
        "is_exact_dominant": (
            most_frequent_bitstring
            == EXACT_BITSTRING
        ),
        "time_spent_simulator": simulator_time,
    }


In [ ]:

# ============================================================
# 15. EXECUÇÃO DE UMA OTIMIZAÇÃO EM SUBESPAÇO ATIVO
# ============================================================

def run_active_cobyla(
    method: str,
    active_indices: Iterable[int],
    restart: int,
    theta_initial_full: np.ndarray,
    maxiter: int,
) -> dict[str, Any]:
    """
    Executa uma rodada completa:
    1. congela os theta fora do conjunto ativo;
    2. otimiza somente os theta ativos;
    3. reconstrói os 30 theta;
    4. usa assign_parameters;
    5. mede 4096 shots;
    6. salva energia, nfev, bitstring e tempos.
    """
    global callback_list

    active_indices = sorted(
        set(
            int(index)
            for index
            in active_indices
        )
    )

    theta_initial_full = np.asarray(
        theta_initial_full,
        dtype=float,
    ).reshape(-1)

    if theta_initial_full.size != n_parameters:
        raise ValueError(
            "theta_initial_full precisa possuir 30 valores."
        )

    # Os parâmetros fora de active_indices ficam exatamente
    # nos valores de theta_initial_full.
    fixed_reference = (
        theta_initial_full.copy()
    )

    x0_active = theta_initial_full[
        active_indices
    ].copy()

    callback_list = []

    def restricted_objective(
        theta_active,
    ):
        theta_full = reconstruct_full_theta(
            theta_active=theta_active,
            active_indices=active_indices,
            fixed_reference=fixed_reference,
        )

        # exp_val continua vendo 30 parâmetros.
        return exp_val(
            theta_full
        )

    optimizer = COBYLA(
        maxiter=int(maxiter)
    )

    optimizer_start = perf_counter()

    try:
        result_optimizer = optimizer.minimize(
            fun=restricted_objective,
            x0=x0_active,
        )
    except Exception as exc:
        return {
            "method": method,
            "restart": restart,
            "status": "failed",
            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
        }

    optimizer_time = (
        perf_counter()
        - optimizer_start
    )

    # result_optimizer.x possui somente n_active valores.
    theta_final_full = reconstruct_full_theta(
        theta_active=result_optimizer.x,
        active_indices=active_indices,
        fixed_reference=fixed_reference,
    )

    # Número de avaliações reais da energia.
    nfev_callback = len(
        callback_list
    )

    nfev_result = getattr(
        result_optimizer,
        "nfev",
        np.nan,
    )

    # assign_parameters e shots acontecem aqui.
    sampling = sample_final_solution(
        theta_full=theta_final_full,
        seed_simulator=(
            random_seed
            + 10_000
            * restart
            + len(active_indices)
        ),
    )

    final_energy = float(
        result_optimizer.fun
    )

    energy_gap = abs(
        final_energy
        - EXACT_ENERGY
    )

    return {
        "method": method,
        "restart": restart,
        "status": "ok",
        "n_active": len(active_indices),
        "active_indices": active_indices,
        "theta_initial_full": theta_initial_full,
        "theta_final_full": theta_final_full,
        "theta_active_final": np.asarray(
            result_optimizer.x,
            dtype=float,
        ),
        "objective_function_value": final_energy,
        "energy_gap": energy_gap,
        "is_near_optimal": (
            energy_gap
            < near_optimal_threshold
        ),
        "nfev_callback": nfev_callback,
        "nfev_result": nfev_result,
        "time_spent_optimizer": optimizer_time,
        "callback_list": [
            (
                np.asarray(
                    theta,
                    dtype=float,
                ),
                float(energy),
            )
            for theta, energy
            in callback_list
        ],
        **sampling,
    }


In [ ]:

# ============================================================
# 16. EXECUTAR A SUÍTE PRINCIPAL COM CHECKPOINT
# ============================================================

runs_pickle_path = (
    output_dir
    / "runs.pkl"
)

if runs_pickle_path.exists():
    all_runs = pd.read_pickle(
        runs_pickle_path
    )

    if not isinstance(
        all_runs,
        list,
    ):
        all_runs = []
else:
    all_runs = []

completed_keys = {
    (
        run.get("method"),
        int(run.get("restart")),
    )
    for run in all_runs
    if run.get("status") == "ok"
}

for restart in range(
    n_restarts
):
    theta_initial_full = (
        initial_points[
            restart
        ].copy()
    )

    for method, active_indices in experiments.items():
        key = (
            method,
            restart,
        )

        if key in completed_keys:
            print(
                "pulando checkpoint:",
                key,
            )
            continue

        print()
        print(
            "=" * 78
        )
        print(
            method,
            "| restart",
            restart,
            "| ativos",
            active_indices,
        )
        print(
            "=" * 78
        )

        run = run_active_cobyla(
            method=method,
            active_indices=active_indices,
            restart=restart,
            theta_initial_full=theta_initial_full,
            maxiter=maxiter_main,
        )

        all_runs.append(
            run
        )

        pd.to_pickle(
            all_runs,
            runs_pickle_path,
        )

        if run["status"] == "ok":
            print(
                "nfev:",
                run["nfev_callback"],
                "| gap:",
                f"{run['energy_gap']:.6e}",
                "| P(exato):",
                f"{run['probability_best_answer']:.4f}",
                "| dominante:",
                run["most_frequent_bitstring"],
            )
        else:
            print(
                "falhou:",
                run["error"],
            )


In [ ]:

# ============================================================
# 17. TOP-4 SEGUIDO DE REFINAMENTO COMPLETO
# ============================================================

if run_two_stage:

    for restart in range(
        n_restarts
    ):
        method = (
            "top4_then_full_refinement"
        )

        key = (
            method,
            restart,
        )

        if key in completed_keys:
            print(
                "pulando checkpoint:",
                key,
            )
            continue

        theta_initial_full = (
            initial_points[
                restart
            ].copy()
        )

        stage1 = run_active_cobyla(
            method="two_stage_top4",
            active_indices=top4,
            restart=restart,
            theta_initial_full=theta_initial_full,
            maxiter=maxiter_main,
        )

        if stage1["status"] != "ok":
            all_runs.append(
                {
                    "method": method,
                    "restart": restart,
                    "status": "failed",
                    "error": (
                        "falha no estágio Top-4: "
                        + stage1.get(
                            "error",
                            "erro desconhecido",
                        )
                    ),
                }
            )
            continue

        # O refinamento completo parte da solução Top-4.
        stage2 = run_active_cobyla(
            method=method,
            active_indices=list(
                range(n_parameters)
            ),
            restart=restart,
            theta_initial_full=stage1[
                "theta_final_full"
            ],
            maxiter=maxiter_refinement,
        )

        if stage2["status"] == "ok":
            stage2[
                "nfev_callback"
            ] += stage1[
                "nfev_callback"
            ]

            stage2[
                "time_spent_optimizer"
            ] += stage1[
                "time_spent_optimizer"
            ]

            stage2[
                "callback_list"
            ] = (
                stage1[
                    "callback_list"
                ]
                + stage2[
                    "callback_list"
                ]
            )

            stage2[
                "stage1_top4_energy"
            ] = stage1[
                "objective_function_value"
            ]

        all_runs.append(
            stage2
        )

        pd.to_pickle(
            all_runs,
            runs_pickle_path,
        )


In [ ]:

# ============================================================
# 18. CONVERTER OS RESULTADOS EM TABELAS
# ============================================================

result_rows = []

for run in all_runs:
    row = {
        "method": run.get(
            "method"
        ),
        "restart": run.get(
            "restart"
        ),
        "status": run.get(
            "status"
        ),
        "error": run.get(
            "error"
        ),
    }

    if run.get(
        "status"
    ) == "ok":
        row.update(
            {
                "n_active": run[
                    "n_active"
                ],
                "active_indices": json.dumps(
                    run[
                        "active_indices"
                    ]
                ),
                "objective_function_value": run[
                    "objective_function_value"
                ],
                "energy_gap": run[
                    "energy_gap"
                ],
                "nfev": run[
                    "nfev_callback"
                ],
                "nfev_result": run[
                    "nfev_result"
                ],
                "time_spent_optimizer": run[
                    "time_spent_optimizer"
                ],
                "time_spent_simulator": run[
                    "time_spent_simulator"
                ],
                "probability_best_answer": run[
                    "probability_best_answer"
                ],
                "most_frequent_bitstring": run[
                    "most_frequent_bitstring"
                ],
                "is_exact_dominant": run[
                    "is_exact_dominant"
                ],
                "is_near_optimal": run[
                    "is_near_optimal"
                ],
            }
        )

    result_rows.append(
        row
    )

results_df = pd.DataFrame(
    result_rows
)

ok_df = results_df[
    results_df[
        "status"
    ]
    == "ok"
].copy()

summary_df = (
    ok_df
    .groupby(
        [
            "method",
            "n_active",
        ],
        as_index=False,
    )
    .agg(
        runs=(
            "restart",
            "count",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
        nfev_mean=(
            "nfev",
            "mean",
        ),
        optimizer_time_median=(
            "time_spent_optimizer",
            "median",
        ),
        simulator_time_median=(
            "time_spent_simulator",
            "median",
        ),
        gap_median=(
            "energy_gap",
            "median",
        ),
        gap_mean=(
            "energy_gap",
            "mean",
        ),
        p_exact_median=(
            "probability_best_answer",
            "median",
        ),
        exact_dominant_rate=(
            "is_exact_dominant",
            "mean",
        ),
        near_optimal_rate=(
            "is_near_optimal",
            "mean",
        ),
    )
)

summary_df[
    "exact_dominant_rate_pct"
] = (
    100
    * summary_df[
        "exact_dominant_rate"
    ]
)

summary_df[
    "near_optimal_rate_pct"
] = (
    100
    * summary_df[
        "near_optimal_rate"
    ]
)

results_df.to_csv(
    output_dir
    / "runs.csv",
    index=False,
)

summary_df.to_csv(
    output_dir
    / "summary.csv",
    index=False,
)

display(
    results_df.sort_values(
        [
            "method",
            "restart",
        ]
    )
)

display(
    summary_df.sort_values(
        [
            "gap_median",
            "nfev_median",
        ]
    )
)


In [ ]:

# ============================================================
# 19. CURVA DE DIMENSÃO ATIVA
# ============================================================

dimension_rows = []

if run_dimension_curve:

    active_dimensions = [
        1,
        2,
        4,
        6,
        8,
        12,
        16,
        30,
    ]

    dimension_checkpoint = (
        output_dir
        / "dim.pkl"
    )

    if dimension_checkpoint.exists():
        dimension_runs = pd.read_pickle(
            dimension_checkpoint
        )
    else:
        dimension_runs = []

    dimension_completed = {
        (
            int(run["dimension"]),
            int(run["restart"]),
        )
        for run in dimension_runs
        if run.get("status") == "ok"
    }

    for dimension in active_dimensions:
        active_indices = ranked_indices[
            :dimension
        ]

        for restart in range(
            n_restarts
        ):
            key = (
                dimension,
                restart,
            )

            if key in dimension_completed:
                continue

            run = run_active_cobyla(
                method=f"top_{dimension}",
                active_indices=active_indices,
                restart=restart,
                theta_initial_full=initial_points[
                    restart
                ],
                maxiter=maxiter_main,
            )

            run[
                "dimension"
            ] = dimension

            dimension_runs.append(
                run
            )

            pd.to_pickle(
                dimension_runs,
                dimension_checkpoint,
            )

    for run in dimension_runs:
        if run.get(
            "status"
        ) != "ok":
            continue

        dimension_rows.append(
            {
                "dimension": run[
                    "dimension"
                ],
                "restart": run[
                    "restart"
                ],
                "nfev": run[
                    "nfev_callback"
                ],
                "energy_gap": run[
                    "energy_gap"
                ],
                "probability_best_answer": run[
                    "probability_best_answer"
                ],
                "is_exact_dominant": run[
                    "is_exact_dominant"
                ],
            }
        )

dimension_df = pd.DataFrame(
    dimension_rows
)

if not dimension_df.empty:
    dimension_summary_df = (
        dimension_df
        .groupby(
            "dimension",
            as_index=False,
        )
        .agg(
            nfev_median=(
                "nfev",
                "median",
            ),
            gap_median=(
                "energy_gap",
                "median",
            ),
            p_exact_median=(
                "probability_best_answer",
                "median",
            ),
            exact_rate=(
                "is_exact_dominant",
                "mean",
            ),
        )
    )

    dimension_df.to_csv(
        output_dir
        / "dim.csv",
        index=False,
    )

    dimension_summary_df.to_csv(
        output_dir
        / "dim_summary.csv",
        index=False,
    )

    display(
        dimension_summary_df
    )


In [ ]:

# ============================================================
# 20. GRÁFICOS
# ============================================================

if not summary_df.empty:

    # --------------------------------------------------------
    # Avaliações da função objetivo
    # --------------------------------------------------------
    plot_df = summary_df.sort_values(
        "nfev_median"
    )

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    bars = ax.barh(
        plot_df[
            "method"
        ],
        plot_df[
            "nfev_median"
        ],
    )

    for bar, value in zip(
        bars,
        plot_df[
            "nfev_median"
        ],
    ):
        ax.text(
            bar.get_width(),
            bar.get_y()
            + bar.get_height()
            / 2,
            f" {value:.0f}",
            va="center",
        )

    ax.set_xlabel(
        "Mediana de avaliações da energia"
    )
    ax.set_title(
        "Custo do COBYLA por estratégia"
    )
    ax.grid(
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "g1_nfev.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # Gap energético
    # --------------------------------------------------------
    plot_df = summary_df.sort_values(
        "gap_median"
    )

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    ax.barh(
        plot_df[
            "method"
        ],
        plot_df[
            "gap_median"
        ],
    )

    ax.axvline(
        near_optimal_threshold,
        linestyle="--",
        linewidth=1.5,
        label="limite quase ótimo",
    )

    ax.set_xscale(
        "log"
    )
    ax.set_xlabel(
        "Gap energético mediano"
    )
    ax.set_title(
        "Qualidade energética por estratégia"
    )
    ax.grid(
        axis="x",
        alpha=0.25,
    )
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "g2_gap.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()

    # --------------------------------------------------------
    # Taxa de bitstring exato dominante
    # --------------------------------------------------------
    plot_df = summary_df.sort_values(
        "exact_dominant_rate_pct"
    )

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    bars = ax.barh(
        plot_df[
            "method"
        ],
        plot_df[
            "exact_dominant_rate_pct"
        ],
    )

    for bar, value in zip(
        bars,
        plot_df[
            "exact_dominant_rate_pct"
        ],
    ):
        ax.text(
            bar.get_width(),
            bar.get_y()
            + bar.get_height()
            / 2,
            f" {value:.1f}%",
            va="center",
        )

    ax.set_xlim(
        0,
        105,
    )
    ax.set_xlabel(
        "Execuções com bitstring exato dominante (%)"
    )
    ax.set_title(
        "Recuperação de 1001011000"
    )
    ax.grid(
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "g3_bit.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()


if (
    "dimension_summary_df"
    in globals()
    and not dimension_summary_df.empty
):
    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.plot(
        dimension_summary_df[
            "dimension"
        ],
        dimension_summary_df[
            "gap_median"
        ],
        marker="o",
        linewidth=2,
    )

    ax.axhline(
        near_optimal_threshold,
        linestyle="--",
        linewidth=1.5,
    )

    ax.set_yscale(
        "log"
    )
    ax.set_xlabel(
        "Quantidade de theta ativos"
    )
    ax.set_ylabel(
        "Gap energético mediano"
    )
    ax.set_title(
        "Menor dimensão ativa que preserva a solução"
    )
    ax.grid(
        alpha=0.25,
    )

    fig.tight_layout()
    fig.savefig(
        output_dir
        / "g4_dim.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:

# ============================================================
# 21. DIAGNÓSTICO AUTOMÁTICO DE VIABILIDADE
# ============================================================

def get_summary_row(
    method,
):
    subset = summary_df[
        summary_df[
            "method"
        ]
        == method
    ]

    if subset.empty:
        return None

    return subset.iloc[0]


full_row = get_summary_row(
    "full_30"
)

top4_row = get_summary_row(
    "top4_theta02_03_14_17"
)

bottom4_row = get_summary_row(
    "bottom4_control"
)

random4_row = get_summary_row(
    "random4_control"
)

diagnostic = {
    "status": "insufficient_results",
}

if (
    full_row is not None
    and top4_row is not None
):
    nfev_reduction = (
        1.0
        - top4_row[
            "nfev_median"
        ]
        / full_row[
            "nfev_median"
        ]
    )

    gap_ratio = (
        top4_row[
            "gap_median"
        ]
        / max(
            full_row[
                "gap_median"
            ],
            1e-15,
        )
    )

    exact_rate_difference_pp = (
        100
        * (
            top4_row[
                "exact_dominant_rate"
            ]
            - full_row[
                "exact_dominant_rate"
            ]
        )
    )

    top4_beats_controls = True

    for control_row in [
        bottom4_row,
        random4_row,
    ]:
        if control_row is None:
            continue

        top4_beats_controls &= (
            top4_row[
                "gap_median"
            ]
            <= control_row[
                "gap_median"
            ]
        )

    diagnostic = {
        "status": "evaluated",
        "nfev_reduction_fraction": float(
            nfev_reduction
        ),
        "nfev_reduction_pct": float(
            100
            * nfev_reduction
        ),
        "gap_ratio_top4_over_full": float(
            gap_ratio
        ),
        "exact_rate_difference_pp": float(
            exact_rate_difference_pp
        ),
        "top4_beats_controls_on_gap": bool(
            top4_beats_controls
        ),
        "provisional_viability": bool(
            nfev_reduction > 0
            and gap_ratio <= 2.0
            and exact_rate_difference_pp >= -10.0
            and top4_beats_controls
        ),
    }

print(
    json.dumps(
        diagnostic,
        indent=2,
        ensure_ascii=False,
    )
)

with open(
    output_dir
    / "diag.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        diagnostic,
        file,
        indent=2,
        ensure_ascii=False,
    )



# Como ler o primeiro experimento

O teste inicial responde se um conjunto reduzido consegue otimizar o circuito partindo de um vetor aleatório.

O Top-4 só seria suficiente globalmente se conseguisse usar menos avaliações e manter qualidade semelhante ao COBYLA completo:

$$
N_{\mathrm{fev}}^{\mathrm{Top4}}
<
N_{\mathrm{fev}}^{\mathrm{Full}},
$$

$$
\Delta E_{\mathrm{Top4}}
\approx
\Delta E_{\mathrm{Full}},
$$

$$
P_{\mathrm{Top4}}(x^\star)
\approx
P_{\mathrm{Full}}(x^\star).
$$

Os resultados anteriores mostraram que o Top-4 não constrói sozinho a solução a partir de um fundo aleatório. Por isso, a próxima etapa não repete essa pergunta. Agora testamos:

- recuperação local com o restante do vetor já bem posicionado;
- perturbações angulares de mesma intensidade;
- aquecimento curto com quatro parâmetros seguido de refinamento completo.



# Parte IV — recuperação local sobre fundo quase ótimo

A auditoria confirmou que 100% dos índices mantiveram a mesma posição física no circuito.

Nesta parte, usamos vetores quase ótimos do banco das 10.000 execuções. Os outros 26 parâmetros permanecem em uma configuração boa, enquanto um grupo de quatro parâmetros é retirado e reotimizado.

A sequência é:

1. medir o fundo quase ótimo intacto;
2. substituir somente os quatro parâmetros do grupo;
3. medir o dano produzido;
4. reotimizar apenas esses quatro parâmetros;
5. medir quanto da energia e da probabilidade exata foi recuperado.

Esse teste mede suficiência local. Ele não afirma que quatro parâmetros resolvem o problema inteiro a partir de qualquer ponto inicial.


In [ ]:

# ============================================================
# 22. CONFIGURAÇÃO DOS TESTES FINAIS
# ============================================================

# False usa 3 repetições para validar o código.
# True usa 30 repetições nas comparações pareadas.
followup_robust_mode = False

followup_n_restarts = (
    30
    if followup_robust_mode
    else 3
)

# Recuperação local e perturbação controlada.
local_active_budget = 80

controlled_perturbation_angles = (
    np.pi / 4,
    np.pi / 2,
)

controlled_active_budget = 80

run_near_optimal_background_test = True
run_controlled_perturbation_test = True

# Novo teste de orçamento do aquecimento Top-4.
run_warmup_budget_sweep = True

warmup_total_budget = 250

warmup_budgets = [
    0,
    10,
    20,
    30,
    40,
]

local_background_gap_threshold = (
    near_optimal_threshold
)

prefer_exact_dominant_backgrounds = True

# Pasta curta e separada para n=3 ou n=30 repetições.
followup_output_dir = (
    output_dir
    / "f"
    / f"n{followup_n_restarts}"
)

followup_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Gerador aleatório exclusivo desta etapa.
# Ele produz a mesma sequência quando o notebook é repetido.
followup_rng = np.random.default_rng(
    random_seed + 2026
)

# Cada linha contém os 30 theta iniciais de um restart.
# Todos os métodos usam a mesma linha no mesmo restart.
followup_initial_points = (
    0.5
    * np.pi
    * followup_rng.random(
        size=(
            followup_n_restarts,
            n_parameters,
        )
    )
)

print(
    "modo robusto:",
    followup_robust_mode,
)

print(
    "repetições pareadas:",
    followup_n_restarts,
)

print(
    "orçamentos de aquecimento:",
    warmup_budgets,
)

print(
    "orçamento total:",
    warmup_total_budget,
)

print(
    "resultados:",
    followup_output_dir.resolve(),
)

if (
    os.name == "nt"
    and len(
        str(
            followup_output_dir.resolve()
        )
    ) > 180
):
    warnings.warn(
        "O caminho ainda está longo. "
        "Use VQE_RESULTS_DIR com uma raiz curta, como D:\\vqe_r."
    )



## Correção do `WinError 206`

O erro não foi causado pelo COBYLA. Ele ocorreu porque o caminho absoluto do projeto, somado aos nomes extensos das pastas e arquivos de checkpoint, ultrapassou o comprimento aceito por alguma operação no Windows.

Esta versão usa uma pasta curta:

```text
C:\Users\<usuário>\vqe_r\base
```

e o acompanhamento fica em:

```text
C:\Users\<usuário>\vqe_r\base\f
```

Os arquivos também receberam nomes curtos, como:

```text
runs.pkl
summary.csv
local.pkl
prog.pkl
```

Os checkpoints antigos não são carregados automaticamente porque estão em outro diretório. Isso evita misturar resultados produzidos antes e depois da correção.


In [ ]:

# ============================================================
# DIAGNÓSTICO DE CAMINHOS DO WINDOWS
# ============================================================

def audit_output_path(
    directory,
    filename,
):
    candidate = (
        Path(directory)
        / filename
    ).resolve()

    return {
        "path": str(candidate),
        "length": len(str(candidate)),
        "exists": candidate.exists(),
    }


path_audit_df = pd.DataFrame(
    [
        audit_output_path(
            output_dir,
            "runs.pkl",
        ),
        audit_output_path(
            followup_output_dir,
            "local.pkl",
        ),
        audit_output_path(
            followup_output_dir,
            "ctrl.pkl",
        ),
        audit_output_path(
            followup_output_dir,
            "warm.pkl",
        ),
        audit_output_path(
            output_dir / "nk",
            "bank.pkl",
        ),
    ]
)

display(path_audit_df)

maximum_path_length = int(
    path_audit_df["length"].max()
)

print(
    "maior comprimento previsto:",
    maximum_path_length,
)

if (
    os.name == "nt"
    and maximum_path_length >= 240
):
    raise RuntimeError(
        "O caminho continua próximo do limite do Windows. "
        'Defina os.environ["VQE_RESULTS_DIR"] = r"D:\\vqe_r" '
        "antes de criar os diretórios."
    )


In [ ]:

# ============================================================
# 23. CARREGAR O BANCO DAS 10.000 EXECUÇÕES
# ============================================================

import ast


# Ajuste manualmente somente quando o arquivo estiver em outro local.
manual_theta_bank_path = None

theta_bank_candidates = [
    Path("../results/merged.pkl"),
    Path("results/merged.pkl"),
    Path("../data/processed/merged.pkl"),
    Path("data/processed/merged.pkl"),
    Path("../data/merged.pkl"),
    Path("merged.pkl"),
]

if manual_theta_bank_path is not None:
    theta_bank_path = Path(
        manual_theta_bank_path
    ).resolve()
else:
    theta_bank_path = next(
        (
            candidate.resolve()
            for candidate
            in theta_bank_candidates
            if candidate.exists()
        ),
        None,
    )


def parse_theta_vector(value):
    """
    Converte a célula best_parameters em vetor NumPy.
    Funciona para ndarray, lista, tupla e string de lista.
    """
    if isinstance(
        value,
        np.ndarray,
    ):
        vector = value

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        vector = np.asarray(
            value
        )

    elif isinstance(
        value,
        str,
    ):
        vector = np.asarray(
            ast.literal_eval(
                value
            )
        )

    else:
        vector = np.asarray(
            value
        )

    return np.asarray(
        vector,
        dtype=float,
    ).reshape(-1)


theta_bank_df = None
theta_bank_matrix = None

if run_near_optimal_background_test:

    if theta_bank_path is None:
        raise FileNotFoundError(
            "merged.pkl não foi localizado. "
            "Defina manual_theta_bank_path na célula 23."
        )

    theta_bank_df = pd.read_pickle(
        theta_bank_path
    )

    required_bank_columns = {
        "best_parameters",
        "objective_function_value",
    }

    missing_bank_columns = (
        required_bank_columns
        - set(
            theta_bank_df.columns
        )
    )

    if missing_bank_columns:
        raise KeyError(
            "O banco não possui as colunas: "
            f"{sorted(missing_bank_columns)}"
        )

    parsed_theta = theta_bank_df[
        "best_parameters"
    ].map(
        parse_theta_vector
    )

    valid_theta_mask = parsed_theta.map(
        lambda vector: (
            vector.size
            == n_parameters
            and np.all(
                np.isfinite(
                    vector
                )
            )
        )
    )

    theta_bank_df = theta_bank_df.loc[
        valid_theta_mask
    ].copy()

    theta_bank_matrix = np.vstack(
        parsed_theta.loc[
            valid_theta_mask
        ].to_numpy()
    )

    theta_bank_df[
        "energy_numeric"
    ] = pd.to_numeric(
        theta_bank_df[
            "objective_function_value"
        ],
        errors="coerce",
    )

    theta_bank_df[
        "energy_gap_from_exact"
    ] = np.abs(
        theta_bank_df[
            "energy_numeric"
        ]
        - EXACT_ENERGY
    )

    if (
        "most_frequent_bitstring"
        in theta_bank_df.columns
    ):
        theta_bank_df[
            "bitstring_clean"
        ] = (
            theta_bank_df[
                "most_frequent_bitstring"
            ]
            .astype(str)
            .str.replace(
                " ",
                "",
                regex=False,
            )
        )
    else:
        theta_bank_df[
            "bitstring_clean"
        ] = ""

    theta_bank_df[
        "theta_bank_position"
    ] = np.arange(
        len(
            theta_bank_df
        )
    )

    print(
        "banco localizado:",
        theta_bank_path,
    )

    print(
        "vetores válidos:",
        theta_bank_matrix.shape,
    )

    print(
        "vetores com gap <",
        local_background_gap_threshold,
        ":",
        int(
            (
                theta_bank_df[
                    "energy_gap_from_exact"
                ]
                < local_background_gap_threshold
            ).sum()
        ),
    )


In [ ]:

# ============================================================
# 24. SELECIONAR FUNDOS QUASE ÓTIMOS
# ============================================================

near_backgrounds_df = None
near_background_matrix = None

if run_near_optimal_background_test:

    finite_mask = np.isfinite(
        theta_bank_df[
            "energy_numeric"
        ].to_numpy()
    )

    near_mask = (
        finite_mask
        & (
            theta_bank_df[
                "energy_gap_from_exact"
            ].to_numpy()
            < local_background_gap_threshold
        )
    )

    exact_near_mask = (
        near_mask
        & (
            theta_bank_df[
                "bitstring_clean"
            ].to_numpy()
            == EXACT_BITSTRING
        )
    )

    if (
        prefer_exact_dominant_backgrounds
        and int(
            exact_near_mask.sum()
        )
        >= followup_n_restarts
    ):
        eligible_mask = (
            exact_near_mask
        )
        background_selection_rule = (
            "gap quase ótimo + bitstring exato dominante"
        )
    else:
        eligible_mask = near_mask
        background_selection_rule = (
            "gap quase ótimo"
        )

    eligible_positions = np.flatnonzero(
        eligible_mask
    )

    if (
        eligible_positions.size
        < followup_n_restarts
    ):
        raise RuntimeError(
            "Não existem fundos quase ótimos suficientes. "
            f"Necessários: {followup_n_restarts}; "
            f"disponíveis: {eligible_positions.size}."
        )

    selected_positions = followup_rng.choice(
        eligible_positions,
        size=followup_n_restarts,
        replace=False,
    )

    near_backgrounds_df = (
        theta_bank_df
        .iloc[
            selected_positions
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    near_background_matrix = (
        theta_bank_matrix[
            selected_positions
        ].copy()
    )

    near_backgrounds_df[
        "background_id"
    ] = np.arange(
        followup_n_restarts
    )

    near_backgrounds_df.to_csv(
        followup_output_dir
        / "near_bg.csv",
        index=False,
    )

    print(
        "regra de seleção:",
        background_selection_rule,
    )

    print(
        "fundos selecionados:",
        near_background_matrix.shape,
    )

    display(
        near_backgrounds_df[
            [
                "background_id",
                "energy_numeric",
                "energy_gap_from_exact",
                "bitstring_clean",
            ]
        ]
    )


In [ ]:

# ============================================================
# 25. AVALIAR UM VETOR SEM OTIMIZAÇÃO
# ============================================================

def evaluate_without_optimization(
    method,
    restart,
    theta_full,
    seed_simulator,
    background_id=None,
):
    """
    Mede um vetor completo antes de chamar o COBYLA.

    Isso permite observar separadamente:
    - a qualidade do fundo quase ótimo;
    - o dano produzido ao retirar o Top-4;
    - a recuperação após reotimizar o Top-4.
    """
    global callback_list

    theta_full = np.asarray(
        theta_full,
        dtype=float,
    ).reshape(-1)

    callback_list = []

    energy = float(
        exp_val(
            theta_full
        )
    )

    sampling = sample_final_solution(
        theta_full=theta_full,
        seed_simulator=int(
            seed_simulator
        ),
    )

    return {
        "method": method,
        "restart": int(
            restart
        ),
        "background_id": background_id,
        "status": "ok",
        "n_active": 0,
        "active_indices": [],
        "theta_initial_full": theta_full.copy(),
        "theta_final_full": theta_full.copy(),
        "objective_function_value": energy,
        "energy_gap": abs(
            energy
            - EXACT_ENERGY
        ),
        "is_near_optimal": (
            abs(
                energy
                - EXACT_ENERGY
            )
            < near_optimal_threshold
        ),
        "nfev_callback": 1,
        "nfev_result": 0,
        "time_spent_optimizer": 0.0,
        "callback_list": [
            (
                theta.copy(),
                float(
                    callback_energy
                ),
            )
            for theta, callback_energy
            in callback_list
        ],
        **sampling,
    }


In [ ]:

# ============================================================
# 26. EXPERIMENTO A — FUNDO QUASE ÓTIMO
# ============================================================

local_checkpoint_path = (
    followup_output_dir
    / "local.pkl"
)

if (
    run_near_optimal_background_test
    and local_checkpoint_path.exists()
):
    local_runs = pd.read_pickle(
        local_checkpoint_path
    )

    if not isinstance(
        local_runs,
        list,
    ):
        local_runs = []
else:
    local_runs = []


def local_run_key(
    method,
    restart,
):
    return (
        str(
            method
        ),
        int(
            restart
        ),
    )


local_completed = {
    local_run_key(
        run.get(
            "method"
        ),
        run.get(
            "restart"
        ),
    )
    for run in local_runs
    if run.get(
        "status"
    ) == "ok"
}


def save_local_run(
    run,
):
    key = local_run_key(
        run[
            "method"
        ],
        run[
            "restart"
        ],
    )

    if key not in local_completed:
        local_runs.append(
            run
        )

        local_completed.add(
            key
        )

        pd.to_pickle(
            local_runs,
            local_checkpoint_path,
        )


if run_near_optimal_background_test:

    for restart in range(
        followup_n_restarts
    ):
        good_background = (
            near_background_matrix[
                restart
            ].copy()
        )

        random_start = (
            followup_initial_points[
                restart
            ].copy()
        )

        background_id = int(
            near_backgrounds_df.loc[
                restart,
                "background_id",
            ]
        )

        # ----------------------------------------------------
        # A1. Fundo quase ótimo intacto
        # ----------------------------------------------------
        method = (
            "near_good_background"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = evaluate_without_optimization(
                method=method,
                restart=restart,
                theta_full=good_background,
                seed_simulator=(
                    random_seed
                    + 100_000
                    + restart
                ),
                background_id=background_id,
            )

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A2. Retirar o Top-4 da região boa
        # ----------------------------------------------------
        top4_reset = (
            good_background.copy()
        )

        top4_reset[
            top4
        ] = random_start[
            top4
        ]

        method = (
            "near_top4_reset_before"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = evaluate_without_optimization(
                method=method,
                restart=restart,
                theta_full=top4_reset,
                seed_simulator=(
                    random_seed
                    + 110_000
                    + restart
                ),
                background_id=background_id,
            )

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A3. Otimizar somente o Top-4 sobre o fundo bom
        # ----------------------------------------------------
        method = (
            "near_top4_reoptimized"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = run_active_cobyla(
                method=method,
                active_indices=top4,
                restart=restart,
                theta_initial_full=top4_reset,
                maxiter=local_active_budget,
            )

            run[
                "background_id"
            ] = background_id

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A4. Controle: retirar o Bottom-4
        # ----------------------------------------------------
        bottom4_reset = (
            good_background.copy()
        )

        bottom4_reset[
            bottom4
        ] = random_start[
            bottom4
        ]

        method = (
            "near_bottom4_reset_before"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = evaluate_without_optimization(
                method=method,
                restart=restart,
                theta_full=bottom4_reset,
                seed_simulator=(
                    random_seed
                    + 120_000
                    + restart
                ),
                background_id=background_id,
            )

            save_local_run(
                run
            )

        # ----------------------------------------------------
        # A5. Reotimizar somente o Bottom-4
        # ----------------------------------------------------
        method = (
            "near_bottom4_reoptimized"
        )

        if local_run_key(
            method,
            restart,
        ) not in local_completed:
            run = run_active_cobyla(
                method=method,
                active_indices=bottom4,
                restart=restart,
                theta_initial_full=bottom4_reset,
                maxiter=local_active_budget,
            )

            run[
                "background_id"
            ] = background_id

            save_local_run(
                run
            )

        print(
            "teste local concluído:",
            restart + 1,
            "/",
            followup_n_restarts,
        )


In [ ]:

# ============================================================
# 27. RESUMO PAREADO DO TESTE LOCAL
# ============================================================

local_results_df = pd.DataFrame(
    [
        {
            "method": run.get(
                "method"
            ),
            "restart": run.get(
                "restart"
            ),
            "background_id": run.get(
                "background_id"
            ),
            "status": run.get(
                "status"
            ),
            "energy": run.get(
                "objective_function_value"
            ),
            "energy_gap": run.get(
                "energy_gap"
            ),
            "nfev": run.get(
                "nfev_callback"
            ),
            "p_exact": run.get(
                "probability_best_answer"
            ),
            "dominant_bitstring": run.get(
                "most_frequent_bitstring"
            ),
            "is_exact_dominant": run.get(
                "is_exact_dominant"
            ),
            "is_near_optimal": run.get(
                "is_near_optimal"
            ),
        }
        for run in local_runs
    ]
)

local_results_df.to_csv(
    followup_output_dir
    / "local_results.csv",
    index=False,
)

local_summary_df = (
    local_results_df[
        local_results_df[
            "status"
        ]
        == "ok"
    ]
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        runs=(
            "restart",
            "count",
        ),
        gap_median=(
            "energy_gap",
            "median",
        ),
        gap_mean=(
            "energy_gap",
            "mean",
        ),
        p_exact_median=(
            "p_exact",
            "median",
        ),
        exact_rate=(
            "is_exact_dominant",
            "mean",
        ),
        near_rate=(
            "is_near_optimal",
            "mean",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
    )
)

local_summary_df[
    "exact_rate_pct"
] = (
    100
    * local_summary_df[
        "exact_rate"
    ]
)

local_summary_df.to_csv(
    followup_output_dir
    / "local_summary.csv",
    index=False,
)

display(
    local_summary_df.sort_values(
        "gap_median"
    )
)


def paired_local_comparison(
    before_method,
    after_method,
    label,
):
    before = (
        local_results_df[
            local_results_df[
                "method"
            ]
            == before_method
        ][
            [
                "restart",
                "energy_gap",
                "p_exact",
                "is_exact_dominant",
            ]
        ]
        .rename(
            columns={
                "energy_gap": (
                    "gap_before"
                ),
                "p_exact": (
                    "p_exact_before"
                ),
                "is_exact_dominant": (
                    "exact_before"
                ),
            }
        )
    )

    after = (
        local_results_df[
            local_results_df[
                "method"
            ]
            == after_method
        ][
            [
                "restart",
                "energy_gap",
                "p_exact",
                "is_exact_dominant",
                "nfev",
            ]
        ]
        .rename(
            columns={
                "energy_gap": (
                    "gap_after"
                ),
                "p_exact": (
                    "p_exact_after"
                ),
                "is_exact_dominant": (
                    "exact_after"
                ),
            }
        )
    )

    paired = before.merge(
        after,
        on="restart",
        validate="one_to_one",
    )

    paired[
        "delta_gap_after_minus_before"
    ] = (
        paired[
            "gap_after"
        ]
        - paired[
            "gap_before"
        ]
    )

    paired[
        "delta_p_exact_after_minus_before"
    ] = (
        paired[
            "p_exact_after"
        ]
        - paired[
            "p_exact_before"
        ]
    )

    paired[
        "group"
    ] = label

    return paired


top4_local_paired_df = paired_local_comparison(
    before_method="near_top4_reset_before",
    after_method="near_top4_reoptimized",
    label="top4",
)

bottom4_local_paired_df = paired_local_comparison(
    before_method="near_bottom4_reset_before",
    after_method="near_bottom4_reoptimized",
    label="bottom4",
)

local_paired_df = pd.concat(
    [
        top4_local_paired_df,
        bottom4_local_paired_df,
    ],
    ignore_index=True,
)

local_paired_df.to_csv(
    followup_output_dir
    / "local_pair.csv",
    index=False,
)

print(
    "Mediana da mudança do gap após reotimização:"
)

display(
    local_paired_df.groupby(
        "group",
        as_index=False,
    ).agg(
        median_delta_gap=(
            "delta_gap_after_minus_before",
            "median",
        ),
        median_delta_p_exact=(
            "delta_p_exact_after_minus_before",
            "median",
        ),
        exact_rate_after=(
            "exact_after",
            "mean",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
    )
)


# Parte V — perturbação angular controlada

No teste anterior, os quatro parâmetros foram substituídos por valores aleatórios. Isso pode deslocar um grupo mais do que outro.

Agora todos os grupos recebem exatamente a mesma intensidade de perturbação angular.

Para um grupo com quatro parâmetros, aplicamos:

$$
\theta_j'
=
\operatorname{wrap}
\left(
\theta_j+s_j\delta
\right),
$$

onde:

- $s_j$ vale $+1$ ou $-1$;
- $\delta$ vale $\pi/4$ ou $\pi/2$;
- `wrap` leva o ângulo para o intervalo $[-\pi,\pi)$.

Como os quatro parâmetros recebem deslocamentos de módulo $\delta$, a intensidade total da intervenção é:

$$
\|\Delta\boldsymbol{\theta}\|_2
=
2\delta.
$$

A mesma sequência de sinais é aplicada ao Top-4, Bottom-4 e Random-4 no mesmo restart.


In [ ]:
# ============================================================
# 28. GRUPOS E FUNÇÕES DA PERTURBAÇÃO CONTROLADA
# ============================================================

controlled_groups = {
    "top4": list(top4),
    "bottom4": list(bottom4),
    "random4": list(random4),
}


def wrap_to_principal_interval(
    theta,
):
    """
    Coloca todos os ângulos no intervalo [-pi, pi).
    """
    theta = np.asarray(
        theta,
        dtype=float,
    )

    return (
        theta
        + np.pi
    ) % (
        2
        * np.pi
    ) - np.pi


def paired_perturbation_signs(
    restart,
    angle_index,
):
    """
    Gera quatro sinais reproduzíveis.

    O mesmo vetor de sinais é usado nos três grupos daquele restart,
    evitando que um grupo receba uma intervenção mais favorável.
    """
    sign_rng = np.random.default_rng(
        random_seed
        + 30_000
        + 100
        * int(restart)
        + int(angle_index)
    )

    return sign_rng.choice(
        [-1.0, 1.0],
        size=4,
        replace=True,
    )


display(
    pd.DataFrame(
        [
            {
                "group": name,
                "indices": indices,
            }
            for name, indices
            in controlled_groups.items()
        ]
    )
)


In [ ]:
# ============================================================
# 29. EXECUTAR A PERTURBAÇÃO CONTROLADA
# ============================================================

controlled_checkpoint_path = (
    followup_output_dir
    / "ctrl.pkl"
)

if (
    run_controlled_perturbation_test
    and controlled_checkpoint_path.exists()
):
    controlled_runs = pd.read_pickle(
        controlled_checkpoint_path
    )

    if not isinstance(
        controlled_runs,
        list,
    ):
        controlled_runs = []
else:
    controlled_runs = []


def controlled_key(
    method,
    restart,
    angle_index,
    group,
):
    return (
        str(method),
        int(restart),
        int(angle_index),
        str(group),
    )


controlled_completed = {
    controlled_key(
        run.get("method"),
        run.get("restart"),
        run.get("angle_index", -1),
        run.get("group", "none"),
    )
    for run in controlled_runs
    if run.get("status") == "ok"
}


def save_controlled_run(
    run,
):
    key = controlled_key(
        run["method"],
        run["restart"],
        run.get("angle_index", -1),
        run.get("group", "none"),
    )

    if key in controlled_completed:
        return

    controlled_runs.append(run)
    controlled_completed.add(key)

    pd.to_pickle(
        controlled_runs,
        controlled_checkpoint_path,
    )


if run_controlled_perturbation_test:

    for restart in range(
        followup_n_restarts
    ):
        good_background = (
            near_background_matrix[
                restart
            ].copy()
        )

        baseline_key = controlled_key(
            "controlled_baseline",
            restart,
            -1,
            "baseline",
        )

        if baseline_key not in controlled_completed:
            baseline_run = evaluate_without_optimization(
                method="controlled_baseline",
                restart=restart,
                theta_full=good_background,
                seed_simulator=(
                    random_seed
                    + 300_000
                    + restart
                ),
                background_id=restart,
            )

            baseline_run["angle_index"] = -1
            baseline_run["angle_delta"] = 0.0
            baseline_run["group"] = "baseline"

            save_controlled_run(
                baseline_run
            )

        for angle_index, angle_delta in enumerate(
            controlled_perturbation_angles
        ):
            signs = paired_perturbation_signs(
                restart=restart,
                angle_index=angle_index,
            )

            for group_name, group_indices in controlled_groups.items():
                perturbed_theta = (
                    good_background.copy()
                )

                perturbed_theta[
                    group_indices
                ] = wrap_to_principal_interval(
                    perturbed_theta[
                        group_indices
                    ]
                    + signs
                    * angle_delta
                )

                before_key = controlled_key(
                    "controlled_before",
                    restart,
                    angle_index,
                    group_name,
                )

                if before_key not in controlled_completed:
                    before_run = evaluate_without_optimization(
                        method="controlled_before",
                        restart=restart,
                        theta_full=perturbed_theta,
                        seed_simulator=(
                            random_seed
                            + 310_000
                            + 1000
                            * angle_index
                            + 10
                            * restart
                            + len(group_indices)
                        ),
                        background_id=restart,
                    )

                    before_run.update(
                        {
                            "angle_index": angle_index,
                            "angle_delta": float(angle_delta),
                            "group": group_name,
                            "active_indices": list(group_indices),
                            "signs": signs.copy(),
                            "circular_l2": float(
                                2
                                * angle_delta
                            ),
                        }
                    )

                    save_controlled_run(
                        before_run
                    )

                after_key = controlled_key(
                    "controlled_after",
                    restart,
                    angle_index,
                    group_name,
                )

                if after_key not in controlled_completed:
                    after_run = run_active_cobyla(
                        method="controlled_after",
                        active_indices=group_indices,
                        restart=restart,
                        theta_initial_full=perturbed_theta,
                        maxiter=controlled_active_budget,
                    )

                    after_run.update(
                        {
                            "background_id": restart,
                            "angle_index": angle_index,
                            "angle_delta": float(angle_delta),
                            "group": group_name,
                            "signs": signs.copy(),
                            "circular_l2": float(
                                2
                                * angle_delta
                            ),
                        }
                    )

                    save_controlled_run(
                        after_run
                    )

        print(
            "perturbação controlada:",
            restart + 1,
            "/",
            followup_n_restarts,
        )


In [ ]:
# ============================================================
# 30. RESUMO PAREADO DA PERTURBAÇÃO CONTROLADA
# ============================================================

controlled_results_df = pd.DataFrame(
    [
        {
            "method": run.get("method"),
            "restart": run.get("restart"),
            "group": run.get("group"),
            "angle_index": run.get("angle_index"),
            "angle_delta": run.get("angle_delta"),
            "circular_l2": run.get("circular_l2"),
            "energy_gap": run.get("energy_gap"),
            "p_exact": run.get(
                "probability_best_answer"
            ),
            "is_exact_dominant": run.get(
                "is_exact_dominant"
            ),
            "nfev": run.get(
                "nfev_callback"
            ),
            "status": run.get("status"),
        }
        for run in controlled_runs
    ]
)

controlled_results_df.to_csv(
    followup_output_dir
    / "ctrl_results.csv",
    index=False,
)

baseline_df = (
    controlled_results_df[
        controlled_results_df["method"]
        == "controlled_baseline"
    ][
        [
            "restart",
            "energy_gap",
            "p_exact",
        ]
    ]
    .rename(
        columns={
            "energy_gap": "gap_baseline",
            "p_exact": "p_exact_baseline",
        }
    )
)

before_df = (
    controlled_results_df[
        controlled_results_df["method"]
        == "controlled_before"
    ][
        [
            "restart",
            "group",
            "angle_index",
            "angle_delta",
            "circular_l2",
            "energy_gap",
            "p_exact",
            "is_exact_dominant",
        ]
    ]
    .rename(
        columns={
            "energy_gap": "gap_before",
            "p_exact": "p_exact_before",
            "is_exact_dominant": "exact_before",
        }
    )
)

after_df = (
    controlled_results_df[
        controlled_results_df["method"]
        == "controlled_after"
    ][
        [
            "restart",
            "group",
            "angle_index",
            "energy_gap",
            "p_exact",
            "is_exact_dominant",
            "nfev",
        ]
    ]
    .rename(
        columns={
            "energy_gap": "gap_after",
            "p_exact": "p_exact_after",
            "is_exact_dominant": "exact_after",
        }
    )
)

controlled_paired_df = (
    before_df
    .merge(
        after_df,
        on=[
            "restart",
            "group",
            "angle_index",
        ],
        validate="one_to_one",
    )
    .merge(
        baseline_df,
        on="restart",
        validate="many_to_one",
    )
)

controlled_paired_df[
    "damage_gap"
] = (
    controlled_paired_df["gap_before"]
    - controlled_paired_df["gap_baseline"]
)

controlled_paired_df[
    "recovered_gap"
] = (
    controlled_paired_df["gap_before"]
    - controlled_paired_df["gap_after"]
)

controlled_paired_df[
    "recovery_fraction"
] = np.where(
    controlled_paired_df["damage_gap"] > 0,
    controlled_paired_df["recovered_gap"]
    / controlled_paired_df["damage_gap"],
    np.nan,
)

controlled_paired_df[
    "delta_p_exact_after_before"
] = (
    controlled_paired_df["p_exact_after"]
    - controlled_paired_df["p_exact_before"]
)

controlled_paired_df.to_csv(
    followup_output_dir
    / "ctrl_pair.csv",
    index=False,
)

controlled_summary_df = (
    controlled_paired_df
    .groupby(
        [
            "group",
            "angle_delta",
        ],
        as_index=False,
    )
    .agg(
        runs=(
            "restart",
            "count",
        ),
        damage_gap_median=(
            "damage_gap",
            "median",
        ),
        gap_after_median=(
            "gap_after",
            "median",
        ),
        recovery_fraction_median=(
            "recovery_fraction",
            "median",
        ),
        delta_p_exact_median=(
            "delta_p_exact_after_before",
            "median",
        ),
        exact_rate_after=(
            "exact_after",
            "mean",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
    )
)

controlled_summary_df[
    "angle_degrees"
] = np.degrees(
    controlled_summary_df[
        "angle_delta"
    ]
)

controlled_summary_df.to_csv(
    followup_output_dir
    / "ctrl_summary.csv",
    index=False,
)

display(
    controlled_summary_df.sort_values(
        [
            "angle_delta",
            "damage_gap_median",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


# Parte VI — quanto tempo o Top-4 deve aquecer o COBYLA?

A configuração anterior usou 40 avaliações no Top-4 e 210 no espaço completo. Agora testamos cinco divisões do mesmo orçamento total.

| Aquecimento Top-4 | Refinamento com 30 theta |
|---:|---:|
| 0 | 250 |
| 10 | 240 |
| 20 | 230 |
| 30 | 220 |
| 40 | 210 |

O caso `0 + 250` é o COBYLA completo.

Além do resultado final, o código registra a primeira avaliação em que a trajetória alcança:

$$
|E-E^\star|<10^{-3}.
$$

Isso permite verificar se algum aquecimento chega ao alvo mais cedo, mesmo quando todos os métodos terminam usando o orçamento completo.


In [ ]:
# ============================================================
# 31. FUNÇÕES DO SWEEP DE AQUECIMENTO
# ============================================================

def first_energy_target_hit(
    callback_rows,
    threshold=near_optimal_threshold,
):
    """
    Retorna a primeira avaliação que atingiu o gap desejado.

    Quando a trajetória nunca atinge o alvo, retorna NaN.
    """
    for evaluation_index, (
        _theta,
        energy,
    ) in enumerate(
        callback_rows,
        start=1,
    ):
        if (
            abs(
                float(energy)
                - EXACT_ENERGY
            )
            < threshold
        ):
            return float(
                evaluation_index
            )

    return np.nan


def optimize_without_sampling(
    active_indices,
    theta_initial_full,
    maxiter,
):
    """
    Executa um estágio do COBYLA sem medir o circuito.

    A medição com 4096 shots é feita somente ao final da estratégia.
    """
    global callback_list

    active_indices = sorted(
        {
            int(index)
            for index
            in active_indices
        }
    )

    theta_initial_full = np.asarray(
        theta_initial_full,
        dtype=float,
    ).reshape(-1)

    fixed_reference = (
        theta_initial_full.copy()
    )

    x0_active = (
        theta_initial_full[
            active_indices
        ].copy()
    )

    callback_list = []

    def stage_objective(
        theta_active,
    ):
        theta_full = reconstruct_full_theta(
            theta_active=theta_active,
            active_indices=active_indices,
            fixed_reference=fixed_reference,
        )

        return exp_val(
            theta_full
        )

    optimizer = COBYLA(
        maxiter=int(maxiter)
    )

    stage_start = perf_counter()

    result = optimizer.minimize(
        fun=stage_objective,
        x0=x0_active,
    )

    stage_time = (
        perf_counter()
        - stage_start
    )

    theta_final_full = reconstruct_full_theta(
        theta_active=result.x,
        active_indices=active_indices,
        fixed_reference=fixed_reference,
    )

    return {
        "theta_final_full": theta_final_full,
        "energy": float(result.fun),
        "energy_gap": abs(
            float(result.fun)
            - EXACT_ENERGY
        ),
        "nfev": len(callback_list),
        "time_s": float(stage_time),
        "callback_list": [
            (
                np.asarray(
                    theta,
                    dtype=float,
                ),
                float(energy),
            )
            for theta, energy
            in callback_list
        ],
    }


def run_warmup_budget_strategy(
    restart,
    theta_initial_full,
    warmup_budget,
):
    """
    Executa uma divisão específica do orçamento.

    warmup_budget = 0:
        250 avaliações diretamente nos 30 theta.

    warmup_budget > 0:
        primeiro otimiza o Top-4;
        depois libera os 30 theta com o orçamento restante.
    """
    warmup_budget = int(
        warmup_budget
    )

    theta_initial_full = np.asarray(
        theta_initial_full,
        dtype=float,
    ).copy()

    if warmup_budget == 0:
        warmup_result = None

        refinement = optimize_without_sampling(
            active_indices=list(
                range(
                    n_parameters
                )
            ),
            theta_initial_full=theta_initial_full,
            maxiter=warmup_total_budget,
        )

        final_theta = (
            refinement[
                "theta_final_full"
            ].copy()
        )

        combined_callback = (
            refinement[
                "callback_list"
            ]
        )

        total_time = float(
            refinement[
                "time_s"
            ]
        )

        nfev_warmup = 0
        nfev_refinement = int(
            refinement[
                "nfev"
            ]
        )

    else:
        warmup_result = optimize_without_sampling(
            active_indices=top4,
            theta_initial_full=theta_initial_full,
            maxiter=warmup_budget,
        )

        remaining_budget = max(
            1,
            warmup_total_budget
            - int(
                warmup_result[
                    "nfev"
                ]
            ),
        )

        refinement = optimize_without_sampling(
            active_indices=list(
                range(
                    n_parameters
                )
            ),
            theta_initial_full=warmup_result[
                "theta_final_full"
            ],
            maxiter=remaining_budget,
        )

        final_theta = (
            refinement[
                "theta_final_full"
            ].copy()
        )

        combined_callback = (
            warmup_result[
                "callback_list"
            ]
            + refinement[
                "callback_list"
            ]
        )

        total_time = (
            float(
                warmup_result[
                    "time_s"
                ]
            )
            + float(
                refinement[
                    "time_s"
                ]
            )
        )

        nfev_warmup = int(
            warmup_result[
                "nfev"
            ]
        )

        nfev_refinement = int(
            refinement[
                "nfev"
            ]
        )

    final_energy = float(
        refinement[
            "energy"
        ]
    )

    sampling = sample_final_solution(
        theta_full=final_theta,
        seed_simulator=(
            random_seed
            + 500_000
            + 1000
            * restart
            + warmup_budget
        ),
    )

    return {
        "method": (
            f"warm_{warmup_budget}"
        ),
        "restart": int(restart),
        "warmup_budget": warmup_budget,
        "refinement_budget_requested": (
            warmup_total_budget
            - warmup_budget
        ),
        "status": "ok",
        "theta_initial_full": theta_initial_full,
        "theta_final_full": final_theta,
        "objective_function_value": final_energy,
        "energy_gap": abs(
            final_energy
            - EXACT_ENERGY
        ),
        "is_near_optimal": (
            abs(
                final_energy
                - EXACT_ENERGY
            )
            < near_optimal_threshold
        ),
        "nfev_callback": len(
            combined_callback
        ),
        "nfev_warmup": nfev_warmup,
        "nfev_refinement": nfev_refinement,
        "first_hit_nfev": first_energy_target_hit(
            combined_callback
        ),
        "time_spent_optimizer": total_time,
        "callback_list": combined_callback,
        **sampling,
    }


In [ ]:
# ============================================================
# 32. EXECUTAR O SWEEP DE ORÇAMENTO
# ============================================================

warmup_checkpoint_path = (
    followup_output_dir
    / "warm.pkl"
)

if (
    run_warmup_budget_sweep
    and warmup_checkpoint_path.exists()
):
    warmup_runs = pd.read_pickle(
        warmup_checkpoint_path
    )

    if not isinstance(
        warmup_runs,
        list,
    ):
        warmup_runs = []
else:
    warmup_runs = []


warmup_completed = {
    (
        int(
            run.get(
                "warmup_budget"
            )
        ),
        int(
            run.get(
                "restart"
            )
        ),
    )
    for run in warmup_runs
    if run.get(
        "status"
    ) == "ok"
}


def save_warmup_run(
    run,
):
    key = (
        int(
            run[
                "warmup_budget"
            ]
        ),
        int(
            run[
                "restart"
            ]
        ),
    )

    if key in warmup_completed:
        return

    warmup_runs.append(
        run
    )

    warmup_completed.add(
        key
    )

    pd.to_pickle(
        warmup_runs,
        warmup_checkpoint_path,
    )


if run_warmup_budget_sweep:

    for restart in range(
        followup_n_restarts
    ):
        theta_start = (
            followup_initial_points[
                restart
            ].copy()
        )

        for warmup_budget in warmup_budgets:
            key = (
                int(
                    warmup_budget
                ),
                int(
                    restart
                ),
            )

            if key in warmup_completed:
                continue

            run = run_warmup_budget_strategy(
                restart=restart,
                theta_initial_full=theta_start,
                warmup_budget=warmup_budget,
            )

            save_warmup_run(
                run
            )

        print(
            "sweep de orçamento:",
            restart + 1,
            "/",
            followup_n_restarts,
        )


In [ ]:
# ============================================================
# 33. RESUMO DO SWEEP DE ORÇAMENTO
# ============================================================

warmup_results_df = pd.DataFrame(
    [
        {
            "warmup_budget": run.get(
                "warmup_budget"
            ),
            "restart": run.get(
                "restart"
            ),
            "energy_gap": run.get(
                "energy_gap"
            ),
            "nfev": run.get(
                "nfev_callback"
            ),
            "nfev_warmup": run.get(
                "nfev_warmup"
            ),
            "nfev_refinement": run.get(
                "nfev_refinement"
            ),
            "first_hit_nfev": run.get(
                "first_hit_nfev"
            ),
            "p_exact": run.get(
                "probability_best_answer"
            ),
            "is_exact_dominant": run.get(
                "is_exact_dominant"
            ),
            "is_near_optimal": run.get(
                "is_near_optimal"
            ),
            "optimizer_time": run.get(
                "time_spent_optimizer"
            ),
        }
        for run in warmup_runs
    ]
)

warmup_results_df.to_csv(
    followup_output_dir
    / "warm_results.csv",
    index=False,
)

warmup_summary_df = (
    warmup_results_df
    .groupby(
        "warmup_budget",
        as_index=False,
    )
    .agg(
        runs=(
            "restart",
            "count",
        ),
        gap_median=(
            "energy_gap",
            "median",
        ),
        gap_mean=(
            "energy_gap",
            "mean",
        ),
        p_exact_median=(
            "p_exact",
            "median",
        ),
        exact_rate=(
            "is_exact_dominant",
            "mean",
        ),
        near_rate=(
            "is_near_optimal",
            "mean",
        ),
        nfev_median=(
            "nfev",
            "median",
        ),
        target_hit_rate=(
            "first_hit_nfev",
            lambda values: float(
                np.isfinite(
                    values
                ).mean()
            ),
        ),
        first_hit_nfev_median=(
            "first_hit_nfev",
            "median",
        ),
        optimizer_time_median=(
            "optimizer_time",
            "median",
        ),
    )
)

warmup_summary_df[
    "refinement_budget"
] = (
    warmup_total_budget
    - warmup_summary_df[
        "warmup_budget"
    ]
)

warmup_summary_df[
    "exact_rate_pct"
] = (
    100
    * warmup_summary_df[
        "exact_rate"
    ]
)

warmup_summary_df[
    "target_hit_rate_pct"
] = (
    100
    * warmup_summary_df[
        "target_hit_rate"
    ]
)

warmup_summary_df.to_csv(
    followup_output_dir
    / "warm_summary.csv",
    index=False,
)

display(
    warmup_summary_df.sort_values(
        [
            "gap_median",
            "warmup_budget",
        ]
    )
)


fig, ax = plt.subplots(
    figsize=(
        9,
        5,
    )
)

ax.plot(
    warmup_summary_df[
        "warmup_budget"
    ],
    warmup_summary_df[
        "gap_median"
    ],
    marker="o",
    linewidth=2,
)

ax.axhline(
    near_optimal_threshold,
    linestyle="--",
    linewidth=1.5,
    label="limite quase ótimo",
)

ax.set_yscale(
    "log"
)

ax.set_xlabel(
    "Avaliações reservadas ao Top-4"
)

ax.set_ylabel(
    "Gap energético mediano"
)

ax.set_title(
    "Efeito do orçamento de aquecimento"
)

ax.grid(
    alpha=0.25
)

ax.legend()

fig.tight_layout()

fig.savefig(
    followup_output_dir
    / "warm_gap.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()


# Parte VII — banco piloto para diferentes valores de $n$ e $k$

## Configurações principais

O primeiro teste mantém dez ativos e altera somente a quantidade escolhida:

```text
n = 10, k = 2
n = 10, k = 3
n = 10, k = 4
n = 10, k = 5
```

Isso separa melhor o efeito de $k$, porque os ativos e os dados continuam os mesmos.

## Variação opcional de $n$

Depois do teste principal, é possível habilitar:

```text
n = 6 e n = 8
```

Nesses casos são usados os primeiros $n$ ativos do mesmo CSV. Portanto, variar $n$ também altera o problema financeiro. Essa parte deve ser interpretada como teste combinado de arquitetura e mudança do conjunto de ativos.

## Tamanho do banco

No modo de teste são executadas poucas rodadas. No modo robusto são produzidos 100 vetores para cada configuração.

Com somente 100 vetores, a análise angular usa menos faixas do que a auditoria das 10.000 execuções.


In [ ]:
# ============================================================
# 34. CONFIGURAÇÃO DO BANCO MULTI-N/K
# ============================================================

# False gera 5 vetores por configuração para validar o fluxo.
# True gera 100 vetores por configuração.
nk_robust_mode = False

nk_runs_per_config = (
    100
    if nk_robust_mode
    else 5
)

# O teste principal pedido: dez ativos escolhendo 2, 3, 4 ou 5.
nk_core_configs = [
    (10, 2),
    (10, 3),
    (10, 4),
    (10, 5),
]

# Habilite depois para verificar se a relação também sobrevive
# quando o número de ativos muda.
run_extra_n_configs = False

nk_extra_configs = [
    (6, 2),
    (6, 3),
    (8, 2),
    (8, 3),
    (8, 4),
]

nk_configs = list(
    nk_core_configs
)

if run_extra_n_configs:
    nk_configs.extend(
        nk_extra_configs
    )

# O banco piloto usa menos iterações que o banco original de 10.000.
# Aumente apenas depois de validar o tempo de execução.
nk_cobyla_maxiter = 150

nk_shots = 4096

# Com 100 vetores, 12 faixas dão aproximadamente oito valores
# por faixa antes do filtro de ocupação mínima.
nk_sensitivity_bins = 12
nk_min_runs_per_bin = 5

nk_output_dir = (
    output_dir
    / "nk"
    / (
        "r100"
        if nk_robust_mode
        else "r5"
    )
)

nk_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "configurações:",
    nk_configs,
)

print(
    "vetores por configuração:",
    nk_runs_per_config,
)

print(
    "parâmetros esperados:",
)

for n_value, k_value in nk_configs:
    n_theta_expected = int(
        k_value
        * (
            2
            * n_value
            - k_value
            - 1
        )
        / 2
    )

    print(
        f"  n={n_value}, k={k_value}:",
        n_theta_expected,
    )


In [ ]:
# ============================================================
# 35. ANSATZ DE DICKE COM MAPA ESTRUTURAL
# ============================================================

def dicke_parameter_count(
    n_value,
    k_value,
):
    """
    Quantidade exata de theta desta construção de Dicke.
    """
    return int(
        k_value
        * (
            2
            * n_value
            - k_value
            - 1
        )
        / 2
    )


def build_tracked_dicke_ansatz(
    n_value,
    k_value,
    seed,
):
    """
    Constrói o mesmo tipo de ansatz e registra onde cada theta aparece.

    O mapa usa:
        l: qubit final do bloco;
        i: qubit inicial;
        d = l - i: distância dentro do bloco;
        gate_type: CY quando d=1, CCY nos demais casos.
    """
    numpy_state = np.random.get_state()

    try:
        np.random.seed(
            int(seed)
        )

        qr = QuantumRegister(
            n_value,
            "q",
        )

        qc = QuantumCircuit(
            qr
        )

        for excitation_index in range(
            k_value
        ):
            qc.x(
                n_value
                - excitation_index
                - 1
            )

        records = []
        aux = 1

        for l_value in range(
            n_value
        )[::-1]:
            for i_value in range(
                l_value - 1,
                l_value - 1 - k_value,
                -1,
            ):
                if i_value >= 0:
                    unique_name = (
                        str(l_value)
                        + str(i_value)
                        + str(aux)
                        + str(
                            np.random.randint(
                                0,
                                int(1e8),
                            )
                        )
                    )

                    if i_value == l_value - 1:
                        gate = CY_parameterized(
                            unique_name
                        )

                        gate_parameter = list(
                            gate.params[0].parameters
                        )[0]

                        qc.cx(
                            qr[i_value],
                            qr[l_value],
                        )

                        qc.append(
                            gate,
                            [
                                qr[i_value],
                                qr[l_value],
                            ],
                        )

                        qc.cx(
                            qr[i_value],
                            qr[l_value],
                        )

                        gate_type = "CY"

                    else:
                        gate = CCY_parameterized(
                            unique_name
                        )

                        gate_parameter = list(
                            gate.params[0].parameters
                        )[0]

                        qc.cx(
                            i_value,
                            l_value,
                        )

                        qc.append(
                            gate,
                            [
                                qr[i_value],
                                qr[i_value + 1],
                                qr[l_value],
                            ],
                        )

                        qc.cx(
                            i_value,
                            l_value,
                        )

                        gate_type = "CCY"

                    records.append(
                        {
                            "parameter_object": gate_parameter,
                            "parameter_name": str(
                                gate_parameter
                            ),
                            "l": int(
                                l_value
                            ),
                            "i": int(
                                i_value
                            ),
                            "distance": int(
                                l_value
                                - i_value
                            ),
                            "gate_type": gate_type,
                        }
                    )

                aux += 1

    finally:
        np.random.set_state(
            numpy_state
        )

    ordered_parameters = list(
        qc.parameters
    )

    parameter_to_index = {
        parameter: index
        for index, parameter
        in enumerate(
            ordered_parameters
        )
    }

    structure_rows = []

    for record in records:
        parameter_object = record.pop(
            "parameter_object"
        )

        theta_index = int(
            parameter_to_index[
                parameter_object
            ]
        )

        structure_rows.append(
            {
                "theta_index": theta_index,
                **record,
            }
        )

    structure_df = (
        pd.DataFrame(
            structure_rows
        )
        .sort_values(
            "theta_index"
        )
        .reset_index(
            drop=True
        )
    )

    expected_count = dicke_parameter_count(
        n_value,
        k_value,
    )

    if (
        len(
            structure_df
        )
        != expected_count
    ):
        raise RuntimeError(
            "Quantidade estrutural incorreta: "
            f"esperado {expected_count}, "
            f"encontrado {len(structure_df)}."
        )

    structure_df[
        "n"
    ] = int(
        n_value
    )

    structure_df[
        "k"
    ] = int(
        k_value
    )

    structure_df[
        "rho_k_over_n"
    ] = (
        k_value
        / n_value
    )

    structure_df[
        "u_l"
    ] = (
        structure_df[
            "l"
        ]
        / max(
            n_value - 1,
            1,
        )
    )

    structure_df[
        "u_distance"
    ] = (
        structure_df[
            "distance"
        ]
        / k_value
    )

    structure_df[
        "u_theta_index"
    ] = (
        structure_df[
            "theta_index"
        ]
        / max(
            expected_count - 1,
            1,
        )
    )

    return (
        qc.decompose(),
        structure_df,
    )


In [ ]:
# ============================================================
# 36. CONSTRUIR UM PROBLEMA PARA CADA n E k
# ============================================================

def build_nk_problem_context(
    n_value,
    k_value,
):
    """
    Cria dados, QUBO, Ising, solução exata e ansatz para uma configuração.
    """
    if n_value > stocks_prices.shape[1]:
        raise ValueError(
            f"O CSV possui somente {stocks_prices.shape[1]} ativos. "
            f"Não é possível usar n={n_value}."
        )

    selected_tickers = list(
        stocks_prices.columns[
            :n_value
        ]
    )

    local_returns = (
        daily_returns[
            selected_tickers
        ]
    )

    local_assets_returns = (
        local_returns.sum()
    )

    local_covariance = (
        local_returns.cov()
    )

    local_covariance_values = (
        local_covariance.to_numpy(
            dtype=float
        )
    )

    local_covariance_values = 0.5 * (
        local_covariance_values
        + local_covariance_values.T
    )

    local_return_values = (
        local_assets_returns.to_numpy(
            dtype=float
        )
    )

    local_model = Model(
        name=(
            f"portfolio_n{n_value}_k{k_value}"
        )
    )

    local_variables = np.array(
        [
            local_model.binary_var(
                name=f"x_{index}"
            )
            for index in range(
                n_value
            )
        ],
        dtype=object,
    )

    local_risk = local_model.sum(
        float(
            local_covariance_values[
                row,
                column,
            ]
        )
        * local_variables[
            row
        ]
        * local_variables[
            column
        ]
        for row in range(
            n_value
        )
        for column in range(
            n_value
        )
    )

    local_return = local_model.sum(
        float(
            local_return_values[
                index
            ]
        )
        * local_variables[
            index
        ]
        for index in range(
            n_value
        )
    )

    local_model.minimize(
        q_value
        * local_risk
        - (
            1
            - q_value
        )
        * local_return
        + risk_free
    )

    local_model.add_constraint(
        local_model.sum(
            local_variables.tolist()
        )
        == k_value,
        ctname="budget",
    )

    local_quadratic_program = (
        from_docplex_mp(
            model=local_model
        )
    )

    local_qubo = (
        QuadraticProgramToQubo()
        .convert(
            local_quadratic_program
        )
    )

    local_ising, local_offset = (
        local_qubo.to_ising()
    )

    local_exact_result = (
        NumPyMinimumEigensolver()
        .compute_minimum_eigenvalue(
            operator=local_ising
        )
    )

    local_exact_energy = float(
        np.real(
            local_exact_result.eigenvalue
            + local_offset
        )
    )

    local_exact_state = (
        local_exact_result
        .eigenstate
        .to_dict()
    )

    local_exact_bitstring = str(
        max(
            local_exact_state,
            key=lambda key: abs(
                local_exact_state[
                    key
                ]
            ) ** 2,
        )
    ).replace(
        " ",
        "",
    )

    local_ansatz, local_structure = (
        build_tracked_dicke_ansatz(
            n_value=n_value,
            k_value=k_value,
            seed=(
                random_seed
                + 100
                * n_value
                + k_value
            ),
        )
    )

    local_n_parameters = int(
        local_ansatz.num_parameters
    )

    expected_parameters = (
        dicke_parameter_count(
            n_value,
            k_value,
        )
    )

    if (
        local_n_parameters
        != expected_parameters
    ):
        raise RuntimeError(
            f"n={n_value}, k={k_value}: "
            f"esperados {expected_parameters} theta, "
            f"encontrados {local_n_parameters}."
        )

    local_simulator = AerSimulator(
        method="statevector",
        device="CPU",
    )

    local_estimator = BackendEstimatorV2(
        backend=local_simulator
    )

    return {
        "n": int(
            n_value
        ),
        "k": int(
            k_value
        ),
        "tickers": selected_tickers,
        "assets_returns": local_assets_returns,
        "covariance": local_covariance,
        "ising": local_ising,
        "offset": float(
            local_offset
        ),
        "exact_energy": local_exact_energy,
        "exact_bitstring": local_exact_bitstring,
        "ansatz": local_ansatz,
        "n_parameters": local_n_parameters,
        "structure_df": local_structure,
        "simulator": local_simulator,
        "estimator": local_estimator,
    }


In [ ]:
# ============================================================
# 37. OTIMIZAR UMA RODADA DO BANCO MULTI-N/K
# ============================================================

def run_nk_cobyla(
    context,
    run_index,
):
    """
    Otimiza uma configuração específica e devolve um vetor theta.
    """
    local_rng = np.random.default_rng(
        random_seed
        + 1_000_000
        + 10_000
        * context[
            "n"
        ]
        + 100
        * context[
            "k"
        ]
        + int(
            run_index
        )
    )

    theta_initial = (
        0.5
        * np.pi
        * local_rng.random(
            context[
                "n_parameters"
            ]
        )
    )

    local_callback = []

    def local_exp_val(
        theta,
    ):
        theta = np.asarray(
            theta,
            dtype=float,
        ).reshape(-1)

        result = context[
            "estimator"
        ].run(
            pubs=[
                (
                    context[
                        "ansatz"
                    ],
                    context[
                        "ising"
                    ],
                    theta,
                )
            ]
        ).result()

        energy = float(
            np.real(
                np.asarray(
                    result[
                        0
                    ].data.evs
                ).reshape(-1)[
                    0
                ]
            )
            + context[
                "offset"
            ]
        )

        local_callback.append(
            (
                theta.copy(),
                energy,
            )
        )

        return energy

    optimizer = COBYLA(
        maxiter=nk_cobyla_maxiter
    )

    start = perf_counter()

    result = optimizer.minimize(
        fun=local_exp_val,
        x0=theta_initial,
    )

    optimizer_time = (
        perf_counter()
        - start
    )

    theta_final = np.asarray(
        result.x,
        dtype=float,
    ).reshape(-1)

    assigned = (
        context[
            "ansatz"
        ]
        .copy()
        .assign_parameters(
            theta_final,
            inplace=False,
        )
    )

    assigned.measure_all()

    simulator_start = perf_counter()

    counts = (
        context[
            "simulator"
        ]
        .run(
            assigned,
            shots=nk_shots,
            seed_simulator=(
                random_seed
                + 2_000_000
                + 10_000
                * context[
                    "n"
                ]
                + 100
                * context[
                    "k"
                ]
                + int(
                    run_index
                )
            ),
        )
        .result()
        .get_counts()
    )

    simulator_time = (
        perf_counter()
        - simulator_start
    )

    counts_clean = {
        str(key).replace(
            " ",
            "",
        ): int(value)
        for key, value
        in counts.items()
    }

    total_counts = int(
        sum(
            counts_clean.values()
        )
    )

    most_frequent = max(
        counts_clean,
        key=counts_clean.get,
    )

    exact_probability = (
        counts_clean.get(
            context[
                "exact_bitstring"
            ],
            0,
        )
        / total_counts
    )

    final_energy = float(
        result.fun
    )

    return {
        "n": context[
            "n"
        ],
        "k": context[
            "k"
        ],
        "run": int(
            run_index
        ),
        "tickers": context[
            "tickers"
        ],
        "n_parameters": context[
            "n_parameters"
        ],
        "initial_point": theta_initial,
        "best_parameters": theta_final,
        "objective_function_value": final_energy,
        "energy_gap": abs(
            final_energy
            - context[
                "exact_energy"
            ]
        ),
        "exact_energy": context[
            "exact_energy"
        ],
        "exact_bitstring": context[
            "exact_bitstring"
        ],
        "probability_best_answer": exact_probability,
        "most_frequent_bitstring": most_frequent,
        "is_exact_dominant": (
            most_frequent
            == context[
                "exact_bitstring"
            ]
        ),
        "nfev": len(
            local_callback
        ),
        "optimizer_time": float(
            optimizer_time
        ),
        "simulator_time": float(
            simulator_time
        ),
        "status": "ok",
    }


In [ ]:
# ============================================================
# 38. GERAR OU RETOMAR OS BANCOS
# ============================================================

nk_contexts = {}

for n_value, k_value in nk_configs:
    print(
        f"construindo contexto n={n_value}, k={k_value}"
    )

    nk_contexts[
        (
            n_value,
            k_value,
        )
    ] = build_nk_problem_context(
        n_value=n_value,
        k_value=k_value,
    )


for (
    n_value,
    k_value,
), context in nk_contexts.items():

    config_dir = (
        nk_output_dir
        / f"n{n_value}k{k_value}"
    )

    config_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_path = (
        config_dir
        / "bank.pkl"
    )

    if checkpoint_path.exists():
        config_runs = pd.read_pickle(
            checkpoint_path
        )

        if not isinstance(
            config_runs,
            list,
        ):
            config_runs = []
    else:
        config_runs = []

    completed_runs = {
        int(
            row[
                "run"
            ]
        )
        for row in config_runs
        if row.get(
            "status"
        ) == "ok"
    }

    for run_index in range(
        nk_runs_per_config
    ):
        if run_index in completed_runs:
            continue

        result_row = run_nk_cobyla(
            context=context,
            run_index=run_index,
        )

        config_runs.append(
            result_row
        )

        pd.to_pickle(
            config_runs,
            checkpoint_path,
        )

        print(
            f"n={n_value}, k={k_value}",
            f"run {run_index + 1}/{nk_runs_per_config}",
            f"gap={result_row['energy_gap']:.6e}",
            f"P*={result_row['probability_best_answer']:.4f}",
        )

    pd.DataFrame(
        config_runs
    ).to_pickle(
        config_dir
        / "bank_df.pkl"
    )


In [ ]:
# ============================================================
# 39. ANÁLISE DE SENSIBILIDADE DE UM BANCO
# ============================================================

def angular_sensitivity_from_bank(
    bank_df,
    structure_df,
    n_bins,
    min_runs_per_bin,
):
    """
    Mede quanto cada theta separa regiões boas e ruins.

    O score combina:
        variação do gap energético;
        variação da taxa do bitstring exato dominante.
    """
    theta_matrix = np.vstack(
        bank_df[
            "best_parameters"
        ].map(
            lambda value: np.asarray(
                value,
                dtype=float,
            ).reshape(-1)
        )
    )

    wrapped_theta = (
        theta_matrix
        + np.pi
    ) % (
        2
        * np.pi
    ) - np.pi

    energy_gap = pd.to_numeric(
        bank_df[
            "energy_gap"
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    exact_flag = (
        bank_df[
            "is_exact_dominant"
        ]
        .astype(bool)
        .to_numpy()
    )

    bin_edges = np.linspace(
        -np.pi,
        np.pi,
        int(
            n_bins
        )
        + 1,
    )

    rows = []

    for theta_index in range(
        theta_matrix.shape[
            1
        ]
    ):
        bin_index = np.digitize(
            wrapped_theta[
                :,
                theta_index,
            ],
            bin_edges[
                1:-1
            ],
            right=False,
        )

        valid_bin_rows = []

        for local_bin in range(
            int(
                n_bins
            )
        ):
            mask = (
                bin_index
                == local_bin
            )

            n_local = int(
                mask.sum()
            )

            if (
                n_local
                < min_runs_per_bin
            ):
                continue

            valid_bin_rows.append(
                {
                    "n": n_local,
                    "gap_median": float(
                        np.nanmedian(
                            energy_gap[
                                mask
                            ]
                        )
                    ),
                    "exact_rate": float(
                        np.mean(
                            exact_flag[
                                mask
                            ]
                        )
                    ),
                }
            )

        if len(
            valid_bin_rows
        ) < 2:
            gap_span = np.nan
            exact_span = np.nan
            valid_bins = len(
                valid_bin_rows
            )
        else:
            local_df = pd.DataFrame(
                valid_bin_rows
            )

            gap_span = float(
                local_df[
                    "gap_median"
                ].max()
                - local_df[
                    "gap_median"
                ].min()
            )

            exact_span = float(
                local_df[
                    "exact_rate"
                ].max()
                - local_df[
                    "exact_rate"
                ].min()
            )

            valid_bins = len(
                local_df
            )

        rows.append(
            {
                "theta_index": int(
                    theta_index
                ),
                "valid_bins": int(
                    valid_bins
                ),
                "gap_span": gap_span,
                "exact_rate_span": exact_span,
            }
        )

    sensitivity_df = pd.DataFrame(
        rows
    )

    for source_column, target_column in [
        (
            "gap_span",
            "gap_score",
        ),
        (
            "exact_rate_span",
            "exact_score",
        ),
    ]:
        values = sensitivity_df[
            source_column
        ].to_numpy(
            dtype=float
        )

        finite = np.isfinite(
            values
        )

        normalized = np.zeros_like(
            values,
            dtype=float,
        )

        if finite.any():
            value_min = np.nanmin(
                values[
                    finite
                ]
            )

            value_max = np.nanmax(
                values[
                    finite
                ]
            )

            if (
                value_max
                > value_min
            ):
                normalized[
                    finite
                ] = (
                    values[
                        finite
                    ]
                    - value_min
                ) / (
                    value_max
                    - value_min
                )

        sensitivity_df[
            target_column
        ] = normalized

    sensitivity_df[
        "combined_score"
    ] = (
        0.5
        * sensitivity_df[
            "gap_score"
        ]
        + 0.5
        * sensitivity_df[
            "exact_score"
        ]
    )

    sensitivity_df[
        "sensitivity_rank"
    ] = (
        sensitivity_df[
            "combined_score"
        ]
        .rank(
            method="min",
            ascending=False,
        )
        .astype(int)
    )

    return (
        sensitivity_df.merge(
            structure_df,
            on="theta_index",
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            "sensitivity_rank"
        )
        .reset_index(
            drop=True
        )
    )


In [ ]:
# ============================================================
# 40. ANALISAR TODOS OS VALORES DE n E k
# ============================================================

nk_sensitivity_tables = {}
nk_top_rows = []

for (
    n_value,
    k_value,
), context in nk_contexts.items():

    config_dir = (
        nk_output_dir
        / f"n{n_value}k{k_value}"
    )

    bank_df = pd.read_pickle(
        config_dir
        / "bank_df.pkl"
    )

    sensitivity_df = (
        angular_sensitivity_from_bank(
            bank_df=bank_df,
            structure_df=context[
                "structure_df"
            ],
            n_bins=nk_sensitivity_bins,
            min_runs_per_bin=nk_min_runs_per_bin,
        )
    )

    sensitivity_df[
        "n"
    ] = int(
        n_value
    )

    sensitivity_df[
        "k"
    ] = int(
        k_value
    )

    sensitivity_df.to_csv(
        config_dir
        / "sensitivity.csv",
        index=False,
    )

    nk_sensitivity_tables[
        (
            n_value,
            k_value,
        )
    ] = sensitivity_df

    top_local = (
        sensitivity_df
        .head(
            min(
                8,
                len(
                    sensitivity_df
                ),
            )
        )
        .copy()
    )

    nk_top_rows.append(
        top_local
    )

    print()
    print(
        f"Top sensibilidade n={n_value}, k={k_value}"
    )

    display(
        top_local[
            [
                "theta_index",
                "sensitivity_rank",
                "combined_score",
                "gap_span",
                "exact_rate_span",
                "gate_type",
                "l",
                "i",
                "distance",
                "u_l",
                "u_distance",
                "u_theta_index",
            ]
        ]
    )

nk_top_sensitivity_df = pd.concat(
    nk_top_rows,
    ignore_index=True,
)

nk_top_sensitivity_df.to_csv(
    nk_output_dir
    / "top_sensitivity_all.csv",
    index=False,
)


In [ ]:
# ============================================================
# 41. REANALISAR O BANCO ORIGINAL DE 10.000 EXECUÇÕES
# ============================================================

original_10k_sensitivity_df = None

if (
    theta_bank_df is not None
    and theta_bank_matrix is not None
):
    original_structure_ansatz, (
        original_structure_df
    ) = build_tracked_dicke_ansatz(
        n_value=10,
        k_value=4,
        seed=random_seed,
    )

    original_bank_for_sensitivity = (
        theta_bank_df.copy()
    )

    original_bank_for_sensitivity[
        "best_parameters"
    ] = [
        row.copy()
        for row in theta_bank_matrix
    ]

    original_bank_for_sensitivity[
        "energy_gap"
    ] = original_bank_for_sensitivity[
        "energy_gap_from_exact"
    ]

    original_bank_for_sensitivity[
        "is_exact_dominant"
    ] = (
        original_bank_for_sensitivity[
            "bitstring_clean"
        ]
        == EXACT_BITSTRING
    )

    original_10k_sensitivity_df = (
        angular_sensitivity_from_bank(
            bank_df=original_bank_for_sensitivity,
            structure_df=original_structure_df,
            n_bins=96,
            min_runs_per_bin=30,
        )
    )

    original_10k_sensitivity_df[
        "n"
    ] = 10

    original_10k_sensitivity_df[
        "k"
    ] = 4

    original_10k_sensitivity_df.to_csv(
        nk_output_dir
        / "original_10k_sensitivity.csv",
        index=False,
    )

    print(
        "Top-10 do banco original:"
    )

    display(
        original_10k_sensitivity_df.head(
            10
        )[
            [
                "theta_index",
                "sensitivity_rank",
                "combined_score",
                "gap_span",
                "exact_rate_span",
                "gate_type",
                "l",
                "i",
                "distance",
                "u_l",
                "u_distance",
                "u_theta_index",
            ]
        ]
    )
else:
    print(
        "O merged.pkl não foi carregado. "
        "A comparação com as 10.000 execuções foi pulada."
    )


# Parte VIII — procurar uma relação estrutural entre os theta

Comparar somente o número do índice não é suficiente porque a quantidade total de parâmetros muda.

Cada parâmetro desta construção pode ser representado por:

$$
(l,d),
$$

onde:

$$
d=l-i.
$$

Também usamos as coordenadas normalizadas:

$$
u_l
=
\frac{l}{n-1},
$$

$$
u_d
=
\frac{d}{k}.
$$

Para levar uma posição sensível de $(n,k)$ para $(n',k')$, usamos:

$$
l'
=
\operatorname{round}
\left[
u_l(n'-1)
\right],
$$

$$
d'
=
\operatorname{round}
\left[
u_dk'
\right].
$$

Depois procuramos no novo circuito o parâmetro mais próximo dessa posição.

Essa relação é uma hipótese estrutural. O banco piloto verifica se o parâmetro previsto aparece entre os mais sensíveis da nova configuração.


In [ ]:
# ============================================================
# 42. MAPEAR POSIÇÕES SENSÍVEIS ENTRE DIFERENTES n E k
# ============================================================

def map_structure_to_target(
    source_row,
    target_structure_df,
):
    """
    Transfere uma coordenada estrutural normalizada para outro circuito.
    """
    target_n = int(
        target_structure_df[
            "n"
        ].iloc[
            0
        ]
    )

    target_k = int(
        target_structure_df[
            "k"
        ].iloc[
            0
        ]
    )

    predicted_l = int(
        np.rint(
            float(
                source_row[
                    "u_l"
                ]
            )
            * (
                target_n
                - 1
            )
        )
    )

    predicted_l = int(
        np.clip(
            predicted_l,
            1,
            target_n - 1,
        )
    )

    predicted_distance = int(
        np.rint(
            float(
                source_row[
                    "u_distance"
                ]
            )
            * target_k
        )
    )

    predicted_distance = int(
        np.clip(
            predicted_distance,
            1,
            min(
                target_k,
                predicted_l,
            ),
        )
    )

    candidates = (
        target_structure_df.copy()
    )

    candidates[
        "mapping_distance"
    ] = (
        (
            candidates[
                "u_l"
            ]
            - float(
                source_row[
                    "u_l"
                ]
            )
        ) ** 2
        + (
            candidates[
                "u_distance"
            ]
            - float(
                source_row[
                    "u_distance"
                ]
            )
        ) ** 2
        + 0.05
        * (
            candidates[
                "gate_type"
            ]
            != source_row[
                "gate_type"
            ]
        ).astype(float)
    )

    exact_pair = candidates[
        (
            candidates[
                "l"
            ]
            == predicted_l
        )
        & (
            candidates[
                "distance"
            ]
            == predicted_distance
        )
    ]

    if not exact_pair.empty:
        selected = exact_pair.sort_values(
            "mapping_distance"
        ).iloc[
            0
        ]
    else:
        selected = candidates.sort_values(
            "mapping_distance"
        ).iloc[
            0
        ]

    return {
        "predicted_l": predicted_l,
        "predicted_distance": predicted_distance,
        "mapped_theta_index": int(
            selected[
                "theta_index"
            ]
        ),
        "mapping_distance": float(
            selected[
                "mapping_distance"
            ]
        ),
    }


mapping_rows = []

# Fonte principal: Top-4 do banco original n=10, k=4.
if original_10k_sensitivity_df is not None:
    source_rows = (
        original_10k_sensitivity_df[
            original_10k_sensitivity_df[
                "theta_index"
            ].isin(
                [
                    2,
                    3,
                    14,
                    17,
                ]
            )
        ]
        .copy()
    )

    for (
        target_n,
        target_k,
    ), target_sensitivity in nk_sensitivity_tables.items():

        target_structure = (
            nk_contexts[
                (
                    target_n,
                    target_k,
                )
            ][
                "structure_df"
            ]
        )

        for _, source_row in source_rows.iterrows():
            mapped = map_structure_to_target(
                source_row=source_row,
                target_structure_df=target_structure,
            )

            mapped_theta = mapped[
                "mapped_theta_index"
            ]

            target_row = target_sensitivity[
                target_sensitivity[
                    "theta_index"
                ]
                == mapped_theta
            ]

            if target_row.empty:
                mapped_rank = np.nan
                mapped_score = np.nan
            else:
                mapped_rank = int(
                    target_row[
                        "sensitivity_rank"
                    ].iloc[
                        0
                    ]
                )

                mapped_score = float(
                    target_row[
                        "combined_score"
                    ].iloc[
                        0
                    ]
                )

            mapping_rows.append(
                {
                    "source_n": 10,
                    "source_k": 4,
                    "source_theta_index": int(
                        source_row[
                            "theta_index"
                        ]
                    ),
                    "source_rank": int(
                        source_row[
                            "sensitivity_rank"
                        ]
                    ),
                    "source_gate_type": source_row[
                        "gate_type"
                    ],
                    "source_u_l": float(
                        source_row[
                            "u_l"
                        ]
                    ),
                    "source_u_distance": float(
                        source_row[
                            "u_distance"
                        ]
                    ),
                    "target_n": int(
                        target_n
                    ),
                    "target_k": int(
                        target_k
                    ),
                    **mapped,
                    "mapped_sensitivity_rank": mapped_rank,
                    "mapped_combined_score": mapped_score,
                    "mapped_is_top4": (
                        bool(
                            mapped_rank
                            <= 4
                        )
                        if np.isfinite(
                            mapped_rank
                        )
                        else False
                    ),
                    "mapped_is_top_quartile": (
                        bool(
                            mapped_rank
                            <= max(
                                1,
                                int(
                                    np.ceil(
                                        len(
                                            target_sensitivity
                                        )
                                        / 4
                                    )
                                ),
                            )
                        )
                        if np.isfinite(
                            mapped_rank
                        )
                        else False
                    ),
                }
            )

structural_mapping_df = pd.DataFrame(
    mapping_rows
)

if not structural_mapping_df.empty:
    structural_mapping_df.to_csv(
        nk_output_dir
        / "structural_mapping.csv",
        index=False,
    )

    display(
        structural_mapping_df
    )

    print(
        "taxa de acerto no Top-4:",
        f"{100 * structural_mapping_df['mapped_is_top4'].mean():.2f}%",
    )

    print(
        "taxa de acerto no quartil superior:",
        f"{100 * structural_mapping_df['mapped_is_top_quartile'].mean():.2f}%",
    )
else:
    print(
        "O mapeamento estrutural depende da análise do banco original."
    )


# Como executar a versão 15

## Etapa 1 — validar rapidamente

Mantenha:

```python
followup_robust_mode = False
nk_robust_mode = False
run_extra_n_configs = False
```

Isso executa:

- três comparações de orçamento;
- cinco vetores para cada valor de $k$;
- somente $n=10$.

## Etapa 2 — banco de 100 vetores por valor de $k$

Depois altere:

```python
nk_robust_mode = True
```

O banco principal terá:

```text
100 vetores para n=10, k=2
100 vetores para n=10, k=3
100 vetores para n=10, k=4
100 vetores para n=10, k=5
```

Os checkpoints ficam separados em `r5` e `r100`.

## Etapa 3 — variar também $n$

Somente depois do teste com $n=10$, altere:

```python
run_extra_n_configs = True
```

A variação de $n$ usa subconjuntos do CSV e deve ser interpretada com mais cuidado.

## Etapa 4 — comparação robusta do aquecimento

Altere:

```python
followup_robust_mode = True
```

O melhor orçamento de aquecimento será escolhido pela combinação de:

- menor gap mediano;
- maior probabilidade do bitstring exato;
- maior taxa de chegada ao alvo;
- menor número mediano da primeira avaliação que alcança o alvo.

## Limite da conclusão

Cem vetores por configuração formam um banco piloto, não substituem as 10.000 execuções. O objetivo inicial é verificar se existe uma tendência estrutural forte o suficiente para justificar uma geração maior.
